Relazione sul progetto

Files knowledge_base.json e orders.json caricati su repository github e scaricati direttamente qui in questo modo la soluzione risulta più pulita senza riportare tutto il testo nelle celle di Colab. Funzione per scaricare una sola volta all'avvio, poi il tool legge dal disco di colab. Più veloce e più robusto invece di fare ogni volta una chiamata a GitHub che invece risulterebbe più lento, dipendente dalla connessione, e durante l'evaluation moltiplicherebbe le richieste inutilmente.

I testi hanno una mediana di 290 caratteri e un massimo di 538, ogni documento è una singola unità di informazione (una scheda, una guida, una policy) quindi spezzarlo ridurrebbe il contesto senza migliorare il retrieval. Ogni dicumento pertanto resta un chunk.

# Step 0 - Requirements

In [1]:
%pip install -q \
    langchain-huggingface \
    langchain-chroma \
    langchain-litellm \
    sentence-transformers \
    litellm \
    ragas \
    "deepeval==4.2.8" \
    "langchain-community==0.4.1" \
    "huggingface-hub>=1.31,<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 765.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 962.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [81]:
import requests
import os
import json
import re
import time
import pandas as pd
import instructor
import asyncio

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.tools import tool

import uuid
from datetime import date, timedelta
from typing import Literal

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

import litellm
from IPython.display import display
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage, SystemMessage
from langchain_litellm import ChatLiteLLM

from typing import Literal
from pydantic import BaseModel, Field, ValidationError

from dataclasses import dataclass, field

# Step 1 - Create KB

In [3]:
BASE_URL = (
    "https://raw.githubusercontent.com/lorenzofranco24/"
    "greenthumb-support-agent/refs/heads/main/data/"
)
DATA_DIR = "data"
DATA_FILES = ["orders.json", "knowledge_base.json", "eval_dataset.json"]
REFERENCE_DATE = "2026-10-04" # Data fissa per rendere riproducibili risposte e test

In [4]:
def download_data_files(file_names: list[str]) -> None:
    """ Download the JSON data files from GitHub and save them in DATA_DIR. """
    os.makedirs(DATA_DIR, exist_ok=True)
    for file_name in file_names:
        response = requests.get(BASE_URL + file_name, timeout=30)
        response.raise_for_status()
        data = response.json()
        with open(f"{DATA_DIR}/{file_name}", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"{file_name} salvato in {DATA_DIR} ✅")

In [5]:
download_data_files(DATA_FILES)

orders.json salvato in data ✅
knowledge_base.json salvato in data ✅
eval_dataset.json salvato in data ✅


In [6]:
with open(f"{DATA_DIR}/orders.json", "r", encoding="utf-8") as f:
    orders = json.load(f)
with open(f"{DATA_DIR}/knowledge_base.json", "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

print(f"Ordini: {len(orders)} | Documenti KB: {len(knowledge_base)}")
print([o for o in orders if o["order_id"] == "GTPO1042"][0])

Ordini: 22 | Documenti KB: 35
{'order_id': 'GTPO1042', 'customer_name': 'Lorenzo A.', 'city': 'Brescia', 'status': 'spedito', 'order_date': '2026-10-02', 'estimated_delivery': '2026-10-05', 'delivered_on': None, 'carrier': 'BRT', 'tracking_code': 'BRT1042IT', 'items': [{'sku': 'BUL-TUL-025', 'name': 'Bulbi di tulipano Darwin Ibrido mix (25)', 'qty': 2, 'unit_price_eur': 12.9}, {'sku': 'CON-BUL-001', 'name': 'Concime per bulbose 1 kg', 'qty': 1, 'unit_price_eur': 8.9}], 'subtotal_eur': 34.7, 'shipping_eur': 5.9, 'total_eur': 40.6, 'notes': None}


# Step 2 - Vectore Store & Tools

## Step 2.0 - Vectore Store: Chroma

In [7]:
EMBEDDING_MODEL = "BAAI/bge-m3"
COLLECTION_NAME = "greenthumb_kb"

In [8]:
def build_documents(raw_kb: dict) -> list[Document]:
  """ Build a list of langchain.Document objects from the knowledge base (1 doc = 1 chunk).
  The title is prepended to the text so it contributes to the embedding.
  """
  return [
      Document(
          page_content=f"{item['title']}\n{item['text']}",
          metadata={
              "id": item["id"],
              "category": item["category"],
              "source": item["source"],
              "title": item["title"]
          },
      )
      for item in raw_kb
  ]


In [9]:
documents = build_documents(knowledge_base)
print(f"Documenti: {len(documents)}")

Documenti: 35


In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    # Normalize vectors to unit length (norm = 1)
    # This allows using Dot Product instead of Cosine Similarity, speeding up vector search.
    encode_kwargs={"normalize_embeddings": True},
)

try:
  vectorstore.delete_collection()
except NameError:
  pass

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"✅ Vector store pronto: {vectorstore._collection.count()} documenti")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Vector store pronto: 35 documenti


### Test calibrazione soglia

Test molto importante per decidere la soglia corretta per giudicare una risposta accettabile o no.
test 1. Il test non mostra una soglia netta che può essere usata come discriminante, il modello paraphrase-multilingual-MiniLM-L12-v2 non sembra portare buoni risultati in questo caso.
test 2. Aggiungo a page_content anche il titolo di ogni documento perché a volte l'informazione più specifica si trova lì. Cambio modello e uso BAAI/bge-m3 che è multilingue (come il precedente) ma è più pesante.
Il cambio ha prodotto risultati migliori, prendiamo come soglia 0.55 per adesso.
Come si pota un bonsai?" prende 0.53, più della carbonara (0.39) e dei trattori (0.42). È normale: la KB parla di potatura di rose, lavanda e olivo, quindi per il modello la query è semanticamente vicina, anche se non c'è nessun documento sui bonsai. Nessun embedding può distinguerlo in modo affidabile, perché la differenza è un fatto ("bonsai") e non un argomento.

In [11]:
test_queries = [
    "Quando si piantano i tulipani?",
    "Quanto costa la spedizione?",
    "Fino a che temperatura resiste l'olivo?",
    "Come si fa la pasta alla carbonara?",
    "Posso restituire una pianta?",
    "Quanto ci mette la consegna?",
    "Come si pota la lavanda?",
    "Vendete trattori?",
    "Come si pota un bonsai?",
]

for query in test_queries:
    results = vectorstore.similarity_search_with_relevance_scores(query, k=3)
    print(f"\nQUERY: {query}")
    for doc, score in results:
        print(f"  {score:.2f} | {doc.metadata['source']}")


QUERY: Quando si piantano i tulipani?
  0.73 | guides/bulbi_autunnali.md
  0.58 | guides/orto_pomodori.md
  0.55 | guides/cura_lavanda.md

QUERY: Quanto costa la spedizione?
  0.69 | policies/spedizioni.md
  0.57 | policies/pagamenti_fatturazione.md
  0.51 | policies/resi.md

QUERY: Fino a che temperatura resiste l'olivo?
  0.59 | products/olivo_leccino.md
  0.58 | guides/olivo_in_vaso.md
  0.49 | products/vaso_terracotta_30.md

QUERY: Come si fa la pasta alla carbonara?
  0.39 | guides/orto_pomodori.md
  0.39 | guides/parassiti_comuni.md
  0.37 | guides/rinvaso_piante.md

QUERY: Posso restituire una pianta?
  0.61 | policies/resi_piante_vive.md
  0.59 | policies/resi.md
  0.55 | policies/spedizioni.md

QUERY: Quanto ci mette la consegna?
  0.68 | policies/spedizioni.md
  0.60 | policies/pagamenti_fatturazione.md
  0.56 | policies/resi.md

QUERY: Come si pota la lavanda?
  0.69 | guides/cura_lavanda.md
  0.55 | products/lavanda_hidcote.md
  0.51 | guides/olivo_in_vaso.md

QUERY: Vende

## Step 2.1 - Tool search_catalog

In [12]:
SCORE_THRESHOLD = 0.55
TOP_K = 3

In [13]:
@tool
def search_catalog(query: str) -> str:
  """Search the company knowledge base (product sheets, growing guides, returns,
    shipping and warranty policies) for information relevant to the question.

    Use this tool for any question about products, prices, planting and plant care,
    shipping times and costs, returns, warranties or escalation rules.
    Do NOT use it for the status of a specific order.

    Args:
        query: the customer's question in natural language, in Italian.

    Returns:
        The most relevant documents, each with its source file, or a message
        stating that no relevant document was found.
    """
  results = vectorstore.similarity_search_with_relevance_scores(
        query, k=TOP_K)
  relevant = [(doc, score) for doc, score in results if score >= SCORE_THRESHOLD]

  if not relevant:
    return "Nessun documento rilevante trovato nella knowledge base."

  blocks = [
      f"[source: {doc.metadata['source']} | score: {score:.2f}]\n{doc.page_content}"
      for doc, score in relevant
  ]
  return "\n\n".join(blocks)




### Test

Il tool funziona come previsto.

Tulipani: la guida giusta è in cima (0.73). Il secondo blocco (orto_pomodori, 0.58) è rumore che ha superato la soglia. Il system prompt dovrà dire all'agente di usare solo le informazioni pertinenti alla domanda e di ignorare il resto. È un caso utile da citare nella relazione.
Lavanda: un solo blocco, quello giusto.
Carbonara e bonsai: messaggio esplicito di nessun risultato. Il bonsai, che aveva il margine più stretto (0.53 contro 0.55), è escluso correttamente.

In [14]:
# Test: query pertinenti e fuori tema
for q in [
    "Quando si piantano i tulipani?",
    "Come si pota la lavanda?",
    "Come si fa la pasta alla carbonara?",
    "Come si pota un bonsai?",
]:
    print(f"\n=== {q} ===")
    print(search_catalog.invoke(q))


=== Quando si piantano i tulipani? ===
[source: guides/bulbi_autunnali.md | score: 0.73]
Guida ai bulbi autunnali (tulipani, narcisi, crocus)
I bulbi a fioritura primaverile come tulipani, narcisi e crocus si piantano in autunno, da ottobre a novembre, quando la temperatura del suolo scende sotto i 10 °C. I tulipani possono essere piantati anche a dicembre se il terreno non è gelato. Prima dell'impianto vanno conservati in luogo fresco, asciutto e ventilato (10-15 °C), mai in frigorifero con la frutta. La profondità di impianto è pari a circa tre volte l'altezza del bulbo (15 cm per i tulipani) e la distanza tra bulbi è di 10 cm. Il lato appuntito va rivolto verso l'alto.

[source: guides/orto_pomodori.md | score: 0.58]
Guida alla coltivazione dei pomodori
Le piantine di pomodoro si trapiantano tra aprile e maggio, dopo le ultime gelate e quando le temperature notturne restano sopra i 10 °C. Distanza tra le piante 50 cm, con tutore di almeno 1,5 metri. Annaffiare alla base, di mattina

## Step 2.2 - Tool get_order_status

In [15]:
# Indice {order_id: ordine}
ORDERS_BY_ID = {o["order_id"]: o for o in orders}

In [16]:
@tool
def get_order_status(order_id: str) -> str:
  """Look up the status of a specific customer order by its order ID.

    Use this tool when the customer asks where an order is, when it will arrive,
    or about a delay, a cancellation, a return or a damaged delivery.
    Do NOT use it for questions about products, plant care or company policies.

    Args:
        order_id: the order ID only, for example GTPO1042.

    Returns:
        Status, dates, tracking code, items, total and any notes for the order,
        or a message stating that the order was not found.
    """
  order_id = order_id.strip().upper()
  order = ORDERS_BY_ID.get(order_id)
  if order is None:
    return f"Ordine {order_id} non trovato."

  items = ", ".join(f"{i['qty']}x {i['name']}" for i in order["items"])
  lines = [
      f"Ordine {order['order_id']}",
      f"Stato: {order['status']}",
      f"Data ordine: {order['order_date']}",
      f"Consegna stimata: {order['estimated_delivery'] or 'Non disponibile'}",
      f"Consegnato il: {order['delivered_on'] or 'Non ancora consegnato'}",
      f"Tracking: {order['tracking_code'] or 'Non ancora disponibile'}",
      f"Prodotti: {items}",
      f"Totale: {order['total_eur']:.2f} €",
  ]
  if order['notes']:
    lines.append(f"Note: {order['notes']}")
  return "\n".join(lines)


### Test

Customer_name e city non compaiono nell'output perché il tool non ha alcuna autenticazione e chiunque indovini il numero di un ordine potrebbe leggere i dati di un altro cliente



In [17]:
for test_id in ["GTPO1042", "GTPO1042", "GTPO1034", "GTPO1046", "GTPO1031", "GTPO1041", "GTPO9999"]:
  print(f"\n=== get_order_status({test_id!r})===")
  print(get_order_status.invoke({"order_id": test_id}))


=== get_order_status('GTPO1042')===
Ordine GTPO1042
Stato: spedito
Data ordine: 2026-10-02
Consegna stimata: 2026-10-05
Consegnato il: Non ancora consegnato
Tracking: BRT1042IT
Prodotti: 2x Bulbi di tulipano Darwin Ibrido mix (25), 1x Concime per bulbose 1 kg
Totale: 40.60 €

=== get_order_status('GTPO1042')===
Ordine GTPO1042
Stato: spedito
Data ordine: 2026-10-02
Consegna stimata: 2026-10-05
Consegnato il: Non ancora consegnato
Tracking: BRT1042IT
Prodotti: 2x Bulbi di tulipano Darwin Ibrido mix (25), 1x Concime per bulbose 1 kg
Totale: 40.60 €

=== get_order_status('GTPO1034')===
Ordine GTPO1034
Stato: in_ritardo
Data ordine: 2026-09-25
Consegna stimata: 2026-10-06
Consegnato il: Non ancora consegnato
Tracking: BRT1034IT
Prodotti: 1x Rosa inglese Gertrude Jekyll (vaso 5 L), 1x Vaso in terracotta 30 cm
Totale: 47.30 €
Note: Ritardo del corriere. Consegna originaria stimata 2026-10-01, nuova stima 2026-10-06.

=== get_order_status('GTPO1046')===
Ordine GTPO1046
Stato: consegnato
Data

## Step 2.3 - Tool escalate_to_human

In [18]:
# Registro delle escalation, utile per verificare l'effetto nei test e nell'evaluation
ESCALATION_LOG: list[dict] = []

In [19]:
def next_working_day(from_date: str) -> str:
  """Return the next Monday-Friday date after from_date (ISO format)."""
  day = date.fromisoformat(from_date) + timedelta(days=1)
  while day.weekday() >= 5: # 5 = Sabato, 6 = Domenica
    day += timedelta(days=1)
  return day.isoformat()

In [20]:
@tool
def escalate_to_human(
  reason: str,
  order_id: str | None = None,
  priority: Literal["normal", "high"] = "normal"
) -> str:
  """Hand the conversation over to a human customer support operator.

    Use this tool ONLY when a first-level assistant cannot or must not resolve the case:
    - damage during transport or defective goods reported by the customer
    - refund requests outside the returns policy
    - an order that cannot be found after checking the order ID
    - a customer threatening legal action
    - an order delayed by more than 3 days
    - a disputed amount above 100 EUR
    Do NOT use it for informational questions that the knowledge base can answer.

    Args:
        reason: one short sentence explaining why a human is needed.
        order_id: numeric order ID if the case concerns an order, otherwise omit it.
        priority: "high" for transport damage, legal threats or amounts above
            100 EUR, otherwise "normal".

    Returns:
        A confirmation with the ticket ID and the expected handling time.
    """
  ticket_id = uuid.uuid4().hex[:8]
  ESCALATION_LOG.append(
      {
          "ticket_id": ticket_id,
          "reason": reason,
          "order_id": order_id,
          "priority": priority,
          "created_at": REFERENCE_DATE,
      }
  )
  return (
      f"Segnalazione creata. Ticket: {ticket_id} (priorità {priority}). "
      f"Gli operatori sono disponibili dal lunedì al venerdì, 9:00-18:00. "
      f"Fuori orario la presa in carico avviene entro le 10:00 del "
      f"{next_working_day(REFERENCE_DATE)}"
  )


### Test

In [21]:
print(escalate_to_human.invoke(
    {"reason": "Danno da trasporto su 2 confezioni di bulbi", "order_id": "GTPO1046", "priority": "high"}
))
print(escalate_to_human.invoke({"reason": "Il cliente chiede un rimborso fuori policy"}))

print(f"\nVoci in ESCALATION_LOG: {len(ESCALATION_LOG)}")
for entry in ESCALATION_LOG:
    print(entry)

Segnalazione creata. Ticket: 296be82a (priorità high). Gli operatori sono disponibili dal lunedì al venerdì, 9:00-18:00. Fuori orario la presa in carico avviene entro le 10:00 del 2026-10-05
Segnalazione creata. Ticket: b3aea9bf (priorità normal). Gli operatori sono disponibili dal lunedì al venerdì, 9:00-18:00. Fuori orario la presa in carico avviene entro le 10:00 del 2026-10-05

Voci in ESCALATION_LOG: 2
{'ticket_id': '296be82a', 'reason': 'Danno da trasporto su 2 confezioni di bulbi', 'order_id': 'GTPO1046', 'priority': 'high', 'created_at': '2026-10-04'}
{'ticket_id': 'b3aea9bf', 'reason': 'Il cliente chiede un rimborso fuori policy', 'order_id': None, 'priority': 'normal', 'created_at': '2026-10-04'}


# Step 3 - Memoria

## Gestione della memoria

L'agente utilizza due livelli di memoria, con funzioni distinte.

### Memoria a lungo termine (semantica)

È il vector store (Chroma, metrica coseno) costruito sulla knowledge base aziendale, raggiungibile tramite il tool `search_catalog`. Contiene conoscenza persistente (35 documenti: schede prodotto, guide di coltivazione, policy) ed è interrogata per similarità semantica. Ogni documento è un unico chunk, perché i testi sono brevi (mediana circa 290 caratteri, massimo 538) e ciascuno costituisce un'unità informativa autonoma.

Gli embedding sono generati in locale con `BAAI/bge-m3`. La scelta è guidata da un test di retrieval: il modello iniziale (`paraphrase-multilingual-MiniLM-L12-v2`) non recuperava in modo affidabile query brevi come "Come si pota la lavanda?" (score 0.34, inferiore a quello di una domanda fuori tema). Con `bge-m3` e con il titolo inserito nel testo indicizzato, le query pertinenti ottengono score tra 0.59 e 0.73 e il documento corretto compare in prima posizione. La soglia di similarità è fissata a 0.55 come filtro grossolano. La valutazione finale di pertinenza spetta all'LLM, che dichiara una confidence bassa o richiede un operatore se il contesto non risponde alla domanda.

### Memoria a breve termine (conversazionale)

La conversazione cresce ad ogni turno, e ogni chiamata a un tool aggiunge più messaggi (richiesta, chiamata, risultato, risposta). Prima o poi la storia non entra più nel contesto, oppure fa pagare token inutili. Sono state valutate due strategie.

**Trimming.** Si conservano solo gli ultimi N messaggi e si scarta il resto.
- Pro: deterministico, nessuna chiamata LLM aggiuntiva, facile da testare.
- Contro: dimentica ciò che è stato detto all'inizio della conversazione (ad esempio il numero d'ordine).

**Summarization.** Un LLM riassume i messaggi più vecchi e il riassunto li sostituisce.
- Pro: conserva il senso dell'intera conversazione.
- Contro: costa una chiamata LLM in più, introduce latenza e può omettere o alterare dettagli.

**Strategia adottata: summary buffer (mista).** Il trimming è il meccanismo di base e la summarization gestisce i messaggi che il trimming scarterebbe. La motivazione è lo scenario tipico del supporto clienti: il numero d'ordine compare al primo turno e viene richiamato implicitamente più avanti ("e quando arriva?"). Il solo trimming lo perderebbe. Il costo della summarization resta contenuto, perché la chiamata extra avviene solo quando la soglia viene superata e non ad ogni turno.

| Parametro | Valore |
|---|---|
| Soglia di attivazione (`SUMMARY_TRIGGER`) | 10 messaggi |
| Finestra recente (`KEEP_RECENT`) | 4 messaggi |
| Modello per il riassunto | `groq/openai/gpt-oss-20b`, temperatura 0 |

**Funzionamento.**
1. **Stato separato.** Il riassunto e la lista degli order ID sono variabili di stato distinte dalla lista dei messaggi. Vengono iniettate nel system prompt ad ogni chiamata, così non contano nella soglia e non possono essere scartate dalla finestra.
2. **Taglio sicuro.** La finestra recente inizia sempre da un messaggio del cliente. Il punto di taglio parte da `len(messages) - 4` e arretra fino al primo messaggio del cliente, in modo da non separare mai una chiamata a un tool dal suo risultato (le API rifiutano storie con risultati orfani). Per questo la finestra contiene 4 messaggi o poco più (in pratica 5-9, perché il taglio arretra fino a un messaggio del cliente).

*Con un uso frequente dei tool, un turno aggiunge circa 4 messaggi, quindi la summarization scatta all’incirca una volta ogni due turni. Un primo valore di finestra pari a 6 la attivava a ogni turno: il parametro è stato ridotto dopo averlo osservato nei log*

3. **Riassunto incrementale e strutturato.** Ad ogni attivazione si riassume "riassunto precedente + messaggi in uscita", non l'intera storia, così il costo resta costante. Lo schema è fisso: ordini citati, prodotti/argomenti, problema del cliente, azioni già eseguite (comprese le escalation con numero di ticket), domanda ancora aperta. ID, date e numeri di ticket vanno riportati letteralmente. I risultati di `search_catalog` sono troncati a 200 caratteri nella trascrizione, mentre quelli di `get_order_status` e `escalate_to_human` restano integri, perché contengono stato e ticket.
4. **Estrazione deterministica degli order ID.** Gli ordini hanno formato `GTPO` seguito da 4 cifre. Una espressione regolare (`\bGTPO\d{4}\b`, case-insensitive) estrae gli ID dai messaggi del cliente ad ogni turno, non solo quando scatta il riassunto, e li accumula in una lista senza duplicati. Questa lista è la fonte autoritativa degli ID: viene passata al prompt del riassunto con l'istruzione di copiarla, e non dipende dal modello.
5. **Fallback.** Se la chiamata di summarization fallisce (rate limit, modello non disponibile, parametro non supportato), la memoria degrada a trimming puro: i messaggi vecchi vengono scartati ma gli order ID restano nello stato.

**Verifica.** Cinque test automatici, tutti superati:
1. conversazione corta (sotto soglia): stato invariato e nessuna chiamata al modello;
2. conversazione di 14 messaggi: ridotta a 6, con il riassunto che riporta correttamente l'ordine dei messaggi scartati e la finestra che non contiene risultati di tool orfani;
3. il punto di taglio ricade sempre su un messaggio del cliente, per valori di finestra da 3 a 6;
4. secondo ciclo di riassunto (incrementale): ordini GTPO1042 e GTPO1046 e il ticket 8020006a restano presenti dopo due riassunti consecutivi;
5. fallimento del modello: la lista si accorcia comunque e gli order ID sono conservati.

**Confronto con il solo trimming.** Non è stato eseguito come esperimento separato. La ritenzione dell'ordine dopo il riassunto è verificata dal test 4 (ordini e ticket presenti dopo due riassunti consecutivi) e dalla conversazione di 5 turni della sezione architettura. Nel dataset di valutazione i casi di memoria (`mem_*`) misurano l'effetto complessivo. Un confronto diretto con il trimming resta un lavoro futuro.

**Limiti e rischi.**
- **Formato ID rigido.** Il riconoscimento vale solo per `GTPO` + 4 cifre: varianti come "GTPO-1042" o "ordine 1042" non vengono riconosciute. Il vincolo è voluto, perché rende l'estrazione prevedibile ed evita falsi positivi (anni, prezzi), ma una menzione implicita dell'ordine non viene catturata dalla regex.
- **Il riassunto può contenere imprecisioni.** Nei test ha riportato correttamente ID e ticket, ma resta un testo generato da un modello. Il rischio è mitigato dallo schema fisso e dalla lista degli ID estratta in modo deterministico, ma non eliminato.
- **Degradazione silenziosa.** Il fallback evita blocchi, ma il guasto della summarization è visibile solo in un messaggio di log. In produzione servirebbe un contatore o un allarme.
- **Dipendenza dal provider.** I modelli Llama inizialmente previsti su Groq (`llama-3.1-8b-instant` e `llama-3.3-70b-versatile`) sono stati dismessi il 16 agosto 2026 e sostituiti con modelli `gpt-oss`. I nomi dei modelli sono definiti in costanti all'inizio del notebook per aggiornarli in un solo punto.
- **Complessità e latenza.** Più codice e più casi da testare, e una chiamata aggiuntiva nei turni in cui scatta il riassunto.

## Step 3.1 - Funzione per estrazione ordine

In [22]:
ORDER_ID_PATTERN = re.compile(r"GTPO\d{4}\b", re.IGNORECASE)

In [23]:
def extract_order_ids(text: str) -> list[str]:
  """Extract order IDs (format GTPO + 4 digits) from a text.

    IDs are normalized to uppercase and returned without duplicates,
    in order of first appearance.
    """
  matches = ORDER_ID_PATTERN.findall(text)
  return list(dict.fromkeys(m.upper() for m in matches))


In [24]:
test_cases = [
    ("Dov'è il mio ordine GTPO1042?", ["GTPO1042"]),
    ("GTPO1042 e gtpo1034", ["GTPO1042", "GTPO1034"]),
    ("ordine #GTPO1042", ["GTPO1042"]),
    ("ordine GTPO1042... sì, l'ordine GTPO1042", ["GTPO1042"]),
    ("Ho speso 1299 euro nel 2026", []),
    ("GTPO12345", []),
    ("ordine 1042", []),
    ("GTPO-1042", []),
]

for text, expected in test_cases:
    result = extract_order_ids(text)
    status = "✅" if result == expected else "❌"
    print(f"{status} {text!r} -> {result} (atteso {expected})")

✅ "Dov'è il mio ordine GTPO1042?" -> ['GTPO1042'] (atteso ['GTPO1042'])
✅ 'GTPO1042 e gtpo1034' -> ['GTPO1042', 'GTPO1034'] (atteso ['GTPO1042', 'GTPO1034'])
✅ 'ordine #GTPO1042' -> ['GTPO1042'] (atteso ['GTPO1042'])
✅ "ordine GTPO1042... sì, l'ordine GTPO1042" -> ['GTPO1042'] (atteso ['GTPO1042'])
✅ 'Ho speso 1299 euro nel 2026' -> [] (atteso [])
✅ 'GTPO12345' -> [] (atteso [])
✅ 'ordine 1042' -> [] (atteso [])
✅ 'GTPO-1042' -> [] (atteso [])


## Step 3.2 - Funzione per riassunto

In [25]:
SUMMARY_MODEL = "groq/openai/gpt-oss-20b"
TOOL_RESULT_MAX_CHARS = 200

In [26]:
SUMMARY_PROMPT = """Sei un assistente che mantiene il riassunto di una conversazione di supporto clienti.
Aggiorna il RIASSUNTO PRECEDENTE integrando i NUOVI MESSAGGI. Non riscrivere da zero:
i campi "Ordini citati" e "Azioni già eseguite" sono cumulativi.

Regole:
- Riporta LETTERALMENTE ID d'ordine, date, importi e numeri di ticket.
- Nel campo "Ordini citati" usa esattamente questo elenco: {order_ids}
- Se un campo non ha informazioni scrivi "nessuno". Non inventare.
- Massimo 120 parole, in italiano.
- Rispondi SOLO con lo schema, senza premesse.

Schema:
Ordini citati: ...
Prodotti/argomenti: ...
Problema del cliente: ...
Azioni già eseguite: ...
Domanda ancora aperta: ...

RIASSUNTO PRECEDENTE:
{previous_summary}

NUOVI MESSAGGI:
{transcript}"""

In [27]:
def content_to_text(content) -> str:
  """Convert a message content (string or list of blocks) into plain text."""
  if isinstance(content, str):
    return content
  if isinstance(content, list):
    parts = [
        block if isinstance(block, str) else block.get("text", "")
        for block in content
    ]
    return " ".join(p for p in parts if p)
  return str(content)

In [28]:
def format_transcript(messages: list[BaseMessage]) -> str:
  """Turn a list of LangChain messages into a readable plain-text transcript.

  Knowledge-base search results are truncated; all other tool results
  (order status, escalation tickets) are kept in full.
  """
  lines = []
  for msg in messages:
    text = content_to_text(msg.content).strip()
    if isinstance(msg, HumanMessage):
      lines.append(f"Cliente: {text}")
    elif isinstance(msg, AIMessage):
      if text:
        lines.append(f"Assistente: {text}")
      for call in msg.tool_calls:
        args = json.dumps(call["args"], ensure_ascii=False)
        lines.append(f"Assistente chiama {call['name']} con {args}")
    elif isinstance(msg, ToolMessage):
      if msg.name == "search_catalog" and len(text) > TOOL_RESULT_MAX_CHARS:
        text = text[:TOOL_RESULT_MAX_CHARS] + "..."
      lines.append(f"Risultato {msg.name}: {text}")
  return "\n".join(lines)



In [29]:
RETRY_WAIT_PATTERN = re.compile(r"try again in ([\d.]+)s")


def call_with_backoff(fn, max_attempts: int = 4):
    """Call fn(); on a rate-limit error wait the time suggested by the provider and retry."""
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except litellm.RateLimitError as exc:
            if attempt == max_attempts:
                raise
            match = RETRY_WAIT_PATTERN.search(str(exc))
            wait = float(match.group(1)) + 1 if match else 15 * attempt
            print(f"⏳ Rate limit: attendo {wait:.0f}s (tentativo {attempt}/{max_attempts})")
            time.sleep(wait)

In [30]:
def summarize_messages(messages: list[BaseMessage], previous_summary: str = "") -> str:
    """Update a structured summary with the messages leaving the recent window.

    Order IDs come from a regex, not from the model, and are passed in the
    prompt so that the model copies them literally.
    """
    transcript = format_transcript(messages)
    previous = previous_summary or "Nessun riassunto precedente."
    order_ids = extract_order_ids(f"{previous_summary}\n{transcript}")

    prompt = SUMMARY_PROMPT.format(
        order_ids=", ".join(order_ids) if order_ids else "nessuno",
        previous_summary=previous,
        transcript=transcript,
    )
    response = call_with_backoff(
        lambda: litellm.completion(
            model=SUMMARY_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            reasoning_effort = "low",
            max_tokens=1000,
        )
    )
    return response["choices"][0].message.content.strip()

### Test

In [31]:
# Conversazione finta. Il ticket è scritto a mano per non sporcare ESCALATION_LOG.
fake_ticket = "Segnalazione creata. Ticket: 8020006a (priorità high)."

test_messages = [
    HumanMessage("Dov'è il mio ordine GTPO1042?"),
    AIMessage("", tool_calls=[{"name": "get_order_status", "args": {"order_id": "GTPO1042"}, "id": "c1"}]),
    ToolMessage(get_order_status.invoke({"order_id": "GTPO1042"}), tool_call_id="c1", name="get_order_status"),
    AIMessage("L'ordine GTPO1042 è spedito e arriverà il 2026-10-05."),
    HumanMessage("Posso piantare adesso i bulbi di tulipano?"),
    AIMessage("", tool_calls=[{"name": "search_catalog", "args": {"query": "Quando si piantano i tulipani?"}, "id": "c2"}]),
    ToolMessage(search_catalog.invoke("Quando si piantano i tulipani?"), tool_call_id="c2", name="search_catalog"),
    AIMessage("I tulipani vanno piantati in ottobre-novembre."),
    HumanMessage("Ho un altro ordine, GTPO1046, con due confezioni danneggiate."),
    AIMessage("", tool_calls=[{"name": "escalate_to_human", "args": {"reason": "Danno da trasporto", "order_id": "GTPO1046", "priority": "high"}, "id": "c3"}]),
    ToolMessage(fake_ticket, tool_call_id="c3", name="escalate_to_human"),
    AIMessage("Ho aperto una segnalazione, ticket 8020006a."),
]

print("=== TRASCRIZIONE ===")
print(format_transcript(test_messages))

summary_1 = summarize_messages(test_messages)
print("\n=== RIASSUNTO 1 ===")
print(summary_1)
for expected in ["GTPO1042", "GTPO1046", "8020006a"]:
    assert expected in summary_1, f"Manca {expected} nel primo riassunto"

# Test incrementale: il secondo giro deve conservare i dati del primo
new_messages = [
    HumanMessage("Grazie. Quanto costa la spedizione?"),
    AIMessage("La spedizione costa 5,90 € sotto i 60 €, gratuita da 60 € in su."),
]
summary_2 = summarize_messages(new_messages, previous_summary=summary_1)
print("\n=== RIASSUNTO 2 (incrementale) ===")
print(summary_2)
for expected in ["GTPO1042", "GTPO1046", "8020006a"]:
    assert expected in summary_2, f"Manca {expected} nel secondo riassunto"

print("\n✅ Test superati")

=== TRASCRIZIONE ===
Cliente: Dov'è il mio ordine GTPO1042?
Assistente chiama get_order_status con {"order_id": "GTPO1042"}
Risultato get_order_status: Ordine GTPO1042
Stato: spedito
Data ordine: 2026-10-02
Consegna stimata: 2026-10-05
Consegnato il: Non ancora consegnato
Tracking: BRT1042IT
Prodotti: 2x Bulbi di tulipano Darwin Ibrido mix (25), 1x Concime per bulbose 1 kg
Totale: 40.60 €
Assistente: L'ordine GTPO1042 è spedito e arriverà il 2026-10-05.
Cliente: Posso piantare adesso i bulbi di tulipano?
Assistente chiama search_catalog con {"query": "Quando si piantano i tulipani?"}
Risultato search_catalog: [source: guides/bulbi_autunnali.md | score: 0.73]
Guida ai bulbi autunnali (tulipani, narcisi, crocus)
I bulbi a fioritura primaverile come tulipani, narcisi e crocus si piantano in autunno, da ottobr...
Assistente: I tulipani vanno piantati in ottobre-novembre.
Cliente: Ho un altro ordine, GTPO1046, con due confezioni danneggiate.
Assistente chiama escalate_to_human con {"reason"

## Step 3.3 - Gestione della memoria

In [32]:
SUMMARY_TRIGGER = 10  # sopra questo numero di messaggi scatta il riassunto
KEEP_RECENT = 4       # messaggi recenti che restano integri dopo il riassunto


In [33]:
def find_cut_index(messages: list[BaseMessage], keep_recent: int = KEEP_RECENT) -> int | None:
    """Return the index where the recent window should start.

    The window always starts on a HumanMessage, so a tool call is never separated
    from its result. The cut moves backwards from len(messages) - keep_recent,
    which keeps the window at keep_recent messages or slightly more.
    Returns None if there is nothing useful to cut.
    """
    target = len(messages) - keep_recent
    for i in range(target, 0, -1):
        if isinstance(messages[i], HumanMessage):
            return i
    return None

In [34]:
def update_order_ids(messages: list[BaseMessage], order_ids: list[str]) -> list[str]:
    """Add the order IDs mentioned by the customer to the authoritative list."""
    found = list(order_ids)
    for msg in messages:
        if isinstance(msg, HumanMessage):
            found.extend(extract_order_ids(content_to_text(msg.content)))
    return list(dict.fromkeys(found))


In [35]:
def manage_memory(
    messages: list[BaseMessage],
    summary: str,
    order_ids: list[str],
) -> tuple[list[BaseMessage], str, list[str]]:
    """Apply the summary-buffer strategy to the short-term memory.

    Order IDs are updated at every turn. When the history exceeds SUMMARY_TRIGGER
    messages, older messages are summarized and only the recent window is kept.
    If the summarization call fails, it falls back to plain trimming: the old
    messages are dropped but the order IDs are preserved.
    """
    order_ids = update_order_ids(messages, order_ids)

    if len(messages) <= SUMMARY_TRIGGER:
        return messages, summary, order_ids

    cut = find_cut_index(messages)
    if cut is None:
        return messages, summary, order_ids

    old_messages, recent_messages = messages[:cut], messages[cut:]
    try:
        summary = summarize_messages(old_messages, summary)
    except Exception as exc:  # fallback: trimming puro, il riassunto precedente resta
        print(f"⚠️ Summarization non riuscita ({type(exc).__name__}): trimming puro")
    return recent_messages, summary, order_ids

### Test

In [36]:
# Riusa test_messages (12) e new_messages (2) della cella precedente
long_history = test_messages + new_messages  # 14 messaggi


def has_orphan_tool_messages(messages: list[BaseMessage]) -> bool:
    """True if the history starts with a ToolMessage or a tool result has no matching call."""
    if messages and isinstance(messages[0], ToolMessage):
        return True
    seen_call_ids = set()
    for msg in messages:
        if isinstance(msg, AIMessage):
            seen_call_ids.update(call["id"] for call in msg.tool_calls)
        elif isinstance(msg, ToolMessage) and msg.tool_call_id not in seen_call_ids:
            return True
    return False


# 1. Conversazione corta: nessuna modifica e nessuna chiamata al modello
original_summarize = summarize_messages
calls = {"n": 0}


def counting_summarize(*args, **kwargs):
    calls["n"] += 1
    return original_summarize(*args, **kwargs)


summarize_messages = counting_summarize
short = test_messages[:8]
msgs, summ, ids = manage_memory(short, "", [])
assert msgs == short and summ == "" and calls["n"] == 0
assert ids == ["GTPO1042"]
print("✅ 1. Conversazione corta: invariata, nessuna chiamata al modello")

# 2. Conversazione lunga: finestra di 6 messaggi, riassunto con gli ID vecchi
recent, summary_a, order_ids_a = manage_memory(long_history, "", [])
assert calls["n"] == 1
assert len(recent) == 6 and isinstance(recent[0], HumanMessage)
assert not has_orphan_tool_messages(recent)
assert "GTPO1042" in summary_a
assert order_ids_a == ["GTPO1042", "GTPO1046"]
print(f"✅ 2. Conversazione lunga: {len(long_history)} -> {len(recent)} messaggi")
print(summary_a)

# 3. Il taglio arretra fino al HumanMessage, mai a metà di una coppia chiamata-risultato
for keep in (3, 4, 5, 6):
    assert find_cut_index(long_history, keep) == 8, f"keep_recent={keep}"
assert find_cut_index(long_history, 20) is None
print("✅ 3. Taglio sempre su un HumanMessage (indice 8)")

# 4. Secondo giro: il riassunto precedente e gli ID sopravvivono
extra = [
    HumanMessage("Come si pota la lavanda?"),
    AIMessage("", tool_calls=[{"name": "search_catalog", "args": {"query": "potatura lavanda"}, "id": "c4"}]),
    ToolMessage("[source: guides/cura_lavanda.md | score: 0.69]\nGuida alla cura della lavanda...", tool_call_id="c4", name="search_catalog"),
    AIMessage("La lavanda si pota dopo la fioritura, tra agosto e settembre."),
    HumanMessage("Grazie, ciao!"),
    AIMessage("Prego, buona giornata!"),
]
recent_b, summary_b, order_ids_b = manage_memory(recent + extra, summary_a, order_ids_a)
assert calls["n"] == 2
assert not has_orphan_tool_messages(recent_b)
for expected in ["GTPO1042", "GTPO1046", "8020006a"]:
    assert expected in summary_b, f"Manca {expected} nel secondo giro"
assert order_ids_b == ["GTPO1042", "GTPO1046"]
print("✅ 4. Secondo giro: ID e ticket conservati")
print(summary_b)

summarize_messages = original_summarize  # ripristina la funzione originale

# 5. Fallimento del modello: trimming puro, ma gli ID restano
good_model = SUMMARY_MODEL
SUMMARY_MODEL = "groq/modello-inesistente"
try:
    recent_c, summary_c, order_ids_c = manage_memory(long_history, "", [])
finally:
    SUMMARY_MODEL = good_model
assert len(recent_c) == 6 and summary_c == ""
assert order_ids_c == ["GTPO1042", "GTPO1046"]
print("✅ 5. Fallback su trimming: lista accorciata e ID conservati")

✅ 1. Conversazione corta: invariata, nessuna chiamata al modello
✅ 2. Conversazione lunga: 14 -> 6 messaggi
Ordini citati: GTPO1042  
Prodotti/argomenti: bulbi di tulipano Darwin Ibrido mix, concime per bulbose, guida su quando piantare tulipani  
Problema del cliente: dove è il suo ordine e quando può piantare i bulbi  
Azioni già eseguite: chiamata get_order_status (ordine spedito, consegna prevista 2026-10-05), ricerca catalogo (guida su piantare tulipani in autunno)  
Domanda ancora aperta: nessuno
✅ 3. Taglio sempre su un HumanMessage (indice 8)
✅ 4. Secondo giro: ID e ticket conservati
Ordini citati: GTPO1042, GTPO1046  
Prodotti/argomenti: bulbi di tulipano Darwin Ibrido mix, concime per bulbose, guida su quando piantare tulipani, spedizione e costi di spedizione  
Problema del cliente: dove è il suo ordine e quando può piantare i bulbi; danni da trasporto su GTPO1046  
Azioni già eseguite: chiamata get_order_status (ordine GTPO1042 spedito, consegna prevista 2026-10-05), ricerc

# Step 4 - Agente ReAct

## Architettura dell'agente

### Loop ReAct

L'agente segue il pattern ReAct (ragionamento → azione → osservazione) con un loop scritto a mano, usando LangChain (messaggi, tool, validazione) e LiteLLM come livello di accesso al modello (`ChatLiteLLM`, modello `groq/openai/gpt-oss-120b`, temperatura 0). Ad ogni passo il modello riceve system prompt, storia e osservazioni; se richiede dei tool, vengono eseguiti e i risultati rientrano come `ToolMessage`; se risponde senza tool, il loop termina. Un tetto di 5 iterazioni evita cicli infiniti: raggiunto il limite, una chiamata finale senza tool, con la conversazione appiattita in testo, forza una risposta basata solo sulle informazioni raccolte.

Il loop è stato scritto a mano invece di usare un agente preconfezionato per tre motivi: rende visibile il ragionamento, si integra senza attriti con la gestione della memoria e permette di inserire controlli deterministici attorno alle chiamate al modello.

### Tool

| Tool | Funzione | Note di progetto |
|---|---|---|
| `search_catalog` | RAG sulla knowledge base (k=3, soglia 0.55) | Restituisce una stringa con `source` e score; messaggio esplicito se nulla supera la soglia |
| `get_order_status` | Lettura dello stato ordine da `orders.json` | ID in formato `GTPO` + 4 cifre; non espone nome e città del cliente |
| `escalate_to_human` | Segnalazione a un operatore | Genera un ticket, lo registra in `ESCALATION_LOG`, indica la presa in carico in base a orari e data di riferimento |

Le docstring dei tool sono parte dell'interfaccia: il modello le legge per decidere quando usarli, e per questo indicano anche quando **non** usarli. Gli errori dei tool (tool inesistente, argomenti non validi) tornano al modello come testo, così può correggersi senza interrompere il loop.

### Prompt di sistema

Il prompt contiene la data di riferimento (2026-10-04, fissa per rendere riproducibile la demo), la descrizione dei tool e sette regole: coprire ogni parte della domanda, rispondere solo con informazioni ottenute dai tool, ignorare risultati non pertinenti, i casi di escalation previsti dalla policy, non duplicare un'escalation già aperta, rispondere in italiano. Riassunto e ordini citati vengono iniettati ad ogni turno (vedi sezione memoria).

### Output strutturato

La risposta finale è un JSON con `answer`, `confidence`, `sources`, `needs_human`. La scelta di progetto è **separare la generazione dalla valutazione**:
- `answer` è il testo prodotto dal loop, senza una seconda riscrittura che potrebbe alterarlo;
- `confidence`, `needs_human` e le fonti usate sono compilati da una chiamata separata in JSON mode, validata con Pydantic. Si usa un modello più piccolo (`gpt-oss-20b`).

Due campi sono protetti da controlli deterministici in codice, perché le loro conseguenze sono concrete:
- **`sources`**: intersezione tra le fonti dichiarate dal modello e quelle realmente restituite da `search_catalog` in quel turno. Il modello può scartare i risultati rumorosi ma non citare un file che non ha letto.
- **`needs_human`**: il codice può solo **alzare** il flag. Diventa vero se un ticket è stato effettivamente creato (non solo richiesto) o se l'ordine consultato ha già una pratica in attesa di operatore.

Se la valutazione fallisce, la risposta esce con `confidence: low` e `needs_human: true`: si preferisce un falso allarme verso l'operatore a un'affermazione di sicurezza non verificata. Costo dichiarato: una chiamata LLM in più per turno.

Una prima versione usava `with_structured_output` del framework, ma il provider rifiutava lo schema. La soluzione adottata (JSON mode + validazione Pydantic) è più portabile tra modelli e mantiene lo stesso controllo sullo schema.

### Gestione dei limiti del provider

Il piano gratuito di Groq impone un tetto di 8000 token al minuto per modello. Contromisure: ripetizione automatica con attesa pari al tempo indicato dal provider, modelli diversi per agente e valutazione (quote indipendenti), pause tra le chiamate nei test. I modelli Llama inizialmente previsti sono stati dismessi da Groq il 16 agosto 2026; i nomi dei modelli sono costanti all'inizio del notebook.

### Verifica

Test automatici su:
- selezione dei tool (domande su ordine, KB, escalation, saluto senza tool);
- domanda a due parti con entrambi i tool (esempio della traccia);
- gestione degli errori dei tool e limite di iterazioni con risposta forzata;
- campi `sources` e `needs_human` (ticket creato, pratica già aperta, domanda senza risposta in KB);
- fallback quando la valutazione fallisce;
- conversazione di 5 turni in cui l'ordine citato al primo viene richiamato in modo implicito dopo il riassunto.

### Limiti osservati

- **Dettagli non supportati dal contesto.** Anche a temperatura 0 il modello aggiunge talvolta frasi plausibili ma assenti dai documenti: nella guida sulla lavanda ha scritto che la potatura "favorisce una nuova fioritura più abbondante"; in un'altra prova ha confuso la data dell'ordine con quella di spedizione; ha promesso "sostituzione o rimborso" senza consultare la KB. È il rischio principale per la *faithfulness* ed è l'oggetto della valutazione.
- **Non determinismo.** Lo stesso input può dare risposte diverse tra esecuzioni; un singolo test non basta, serve un dataset.
- **Controllo sulle pratiche aperte basato su testo.** Cerca la frase "in attesa di operatore" nelle note dell'ordine; in un sistema reale sarebbe un campo strutturato.
- **Nessuna autenticazione.** Chiunque conosca un ID ordine ne legge lo stato. Mitigazione parziale: nome e città non vengono esposti.
- **Escalation lasciata al modello.** La regola "ritardo oltre 3 giorni" è nel prompt, non verificata in codice.

**Esito della valutazione rispetto a questi limiti.** I dettagli non supportati dal contesto restano il problema principale: `kb_05` ha faithfulness 0.0 anche dopo l'irrigidimento della regola 2, quindi il prompt da solo non basta. L'escalation lasciata al modello è migliorata con la regola 7 (correctness sui casi di escalation da 0.40 a 0.77), ma non è garantita in codice. Il non determinismo è confermato: due run consecutivi hanno dato 17/18 con un caso diverso a fallire. Risultati completi e tabelle sono nella Relazione Step 5 e nella Relazione finale.

## Step 4.1 - Smoke Test

In [37]:
AGENT_MODEL = "groq/openai/gpt-oss-120b"
ASSESSOR_MODEL = "groq/openai/gpt-oss-20b"  # modello separato: quota token indipendente
MAX_ITERATIONS = 5

In [38]:
TOOLS = [search_catalog, get_order_status, escalate_to_human]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

llm = ChatLiteLLM(model=AGENT_MODEL, temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)

In [39]:
SMOKE_SYSTEM_PROMPT = (
    "Sei un assistente di supporto clienti per un negozio online di giardinaggio. "
    "Usa i tool disponibili quando servono per rispondere."
)

smoke_tests = [
    ("Dov'è il mio ordine GTPO1042?", {"get_order_status"}),
    ("Quando si piantano i tulipani?", {"search_catalog"}),
    ("Quando arriva l'ordine GTPO1042? Posso piantare adesso i bulbi di tulipano?",
     {"get_order_status", "search_catalog"}),
    ("Il mio ordine GTPO1046 è arrivato con due confezioni danneggiate, voglio parlare con un operatore.",
     {"escalate_to_human"}),
    ("Ciao!", set()),
]

for question, expected in smoke_tests:
    ai_message = llm_with_tools.invoke(
        [SystemMessage(SMOKE_SYSTEM_PROMPT), HumanMessage(question)]
    )
    called = {call["name"] for call in ai_message.tool_calls}
    status = "✅" if called == expected else "❌"
    print(f"\n{status} {question}")
    print(f"   attesi: {sorted(expected)} | chiamati: {sorted(called)}")
    for call in ai_message.tool_calls:
        print(f"   -> {call['name']}({call['args']})")
    if ai_message.content:
        print(f"   testo: {ai_message.content[:150]}")


✅ Dov'è il mio ordine GTPO1042?
   attesi: ['get_order_status'] | chiamati: ['get_order_status']
   -> get_order_status({'order_id': 'GTPO1042'})

✅ Quando si piantano i tulipani?
   attesi: ['search_catalog'] | chiamati: ['search_catalog']
   -> search_catalog({'query': 'quando piantare i tulipani'})

❌ Quando arriva l'ordine GTPO1042? Posso piantare adesso i bulbi di tulipano?
   attesi: ['get_order_status', 'search_catalog'] | chiamati: ['get_order_status']
   -> get_order_status({'order_id': 'GTPO1042'})

✅ Il mio ordine GTPO1046 è arrivato con due confezioni danneggiate, voglio parlare con un operatore.
   attesi: ['escalate_to_human'] | chiamati: ['escalate_to_human']
   -> escalate_to_human({'order_id': 'GTPO1046', 'priority': 'high', 'reason': "Confezioni danneggiate all'arrivo, richiede assistenza operatore"})

✅ Ciao!
   attesi: [] | chiamati: []
   testo: Ciao! 😊 Come posso aiutarti oggi?


## Step 4.2 - Modello di risposta e System Prompt

In [73]:
class AgentResponse(BaseModel):
    """Structured final response returned by the support agent."""

    answer: str = Field(description="Risposta al cliente, in italiano, chiara e concisa.")
    confidence: Literal["low", "medium", "high"] = Field(
        description="Affidabilità della risposta: high, medium o low."
    )
    sources: list[str] = Field(
        default_factory=list,
        description="File della knowledge base usati per rispondere.",
    )
    needs_human: bool = Field(description="True se il caso richiede un operatore umano.")


class ResponseAssessment(BaseModel):
    """Model-generated assessment of the final answer (not the answer itself)."""

    confidence: Literal["low", "medium", "high"] = Field(
        description=(
            "'high' se la risposta è interamente supportata dai risultati dei tool; "
            "'medium' se solo in parte; 'low' se le informazioni mancano o sono incerte."
        )
    )
    needs_human: bool = Field(
        description="True se il cliente deve essere preso in carico da un operatore umano."
    )
    used_sources: list[str] = Field(
        default_factory=list,
        description="File 'source' effettivamente usati per la risposta. Vuoto se nessuno.",
    )


SYSTEM_PROMPT_TEMPLATE = """Sei un assistente di supporto clienti di primo livello per un negozio online di giardinaggio.
La data di oggi è {today}: usala per interpretare "oggi", "domani" e le scadenze.

Strumenti:
- search_catalog: prodotti, prezzi, coltivazione, spedizioni, resi, garanzie, regole di escalation.
- get_order_status: stato di un ordine. Passa l'ID ESATTAMENTE come lo scrive il cliente (formato GTPO seguito da 4 cifre). Se il cliente parla di un ordine senza darne l'ID: se in "Ordini citati finora" c'è un solo ordine usa quello; se ce ne sono più di uno o nessuno, chiedi di quale si tratta.
- escalate_to_human: passaggio a un operatore umano.

Regole:
1. Se la domanda ha più parti, usa i tool necessari per OGNI parte prima di rispondere.
2. Rispondi solo con informazioni ottenute dai tool: riporta fatti, valori e passaggi presenti nei risultati, senza aggiungere consigli, dosi, procedure o dettagli tuoi che non compaiono nelle fonti, anche se ti sembrano di buon senso. Se la knowledge base non contiene la risposta, dillo chiaramente senza inventare.
3. Se un risultato di ricerca non è pertinente alla domanda, ignoralo.
4. Escala a un operatore nei casi previsti dalla policy: danni da trasporto, rimborsi fuori policy, ordine non trovato, minacce legali, ordini in ritardo di oltre 3 giorni, importi contestati oltre 100 EUR. Il ritardo si calcola come differenza tra la data di consegna originaria (indicata nelle note dell'ordine) e la nuova data stimata. Una richiesta di rimborso o di indennizzo per un ritardo o per un disservizio è sempre fuori policy, perché la policy dei resi riguarda solo i prodotti non utilizzati restituiti: se il cliente cita un ordine, verifica prima il suo stato con get_order_status, poi escala senza cercare altre regole. Non escalare per domande informative.
5. Non aprire una seconda escalation per un caso già segnalato (controlla "Azioni già eseguite" e le note dell'ordine). Se la pratica è già aperta, informa il cliente che un operatore se ne sta occupando: il caso resta comunque da gestire da un operatore.
6. Rispondi in italiano, in modo cordiale e conciso.
7. Quando escali un caso, o informi che una pratica è già aperta, cita sempre l'ID dell'ordine e il motivo (per esempio il danno da trasporto, oppure il ritardo con la data originaria e la nuova data stimata). Se esiste già una pratica, di' che è già in carico a un operatore invece di dire che l'hai appena segnalata.

Ordini citati finora: {order_ids}

Riassunto della conversazione precedente:
{summary}"""

ASSESSMENT_PROMPT = """Valuta la risposta di un assistente di supporto clienti.

DOMANDA DEL CLIENTE:
{question}

RISULTATI DEI TOOL IN QUESTO TURNO:
{observations}

RISPOSTA DELL'ASSISTENTE:
{answer}

Compila la valutazione. Per used_sources indica solo i file che hanno davvero
contribuito alla risposta, ignorando risultati non pertinenti.

Rispondi SOLO con un oggetto JSON con esattamente questi campi:
{{"confidence": "low" | "medium" | "high", "needs_human": true | false, "used_sources": ["file1", "file2"]}}"""


def build_system_prompt(summary: str, order_ids: list[str]) -> str:
    """Build the system prompt with the reference date, cited orders and summary."""
    return SYSTEM_PROMPT_TEMPLATE.format(
        today=REFERENCE_DATE,
        order_ids=", ".join(order_ids) if order_ids else "nessuno",
        summary=summary or "nessuno (inizio conversazione)",
    )


### Test

In [41]:
# Test 1: risposta valida
valid = AgentResponse(
    answer="L'ordine GTPO1042 arriva domani.",
    confidence="high",
    sources=["guides/bulbi_autunnali.md"],
    needs_human=False,
)
print(valid.model_dump_json(indent=2))

# Test 2: valore di confidence non ammesso
try:
    AgentResponse(answer="ok", confidence="altissima", needs_human=False)
    print("❌ confidence non validata")
except ValidationError:
    print("✅ confidence non valida rifiutata")

# Test 3: prompt senza e con memoria
print("\n=== PROMPT (inizio conversazione) ===")
print(build_system_prompt("", []))
print("\n=== PROMPT (con riassunto) ===")
print(build_system_prompt(summary_b, order_ids_b)[-600:])

{
  "answer": "L'ordine GTPO1042 arriva domani.",
  "confidence": "high",
  "sources": [
    "guides/bulbi_autunnali.md"
  ],
  "needs_human": false
}
✅ confidence non valida rifiutata

=== PROMPT (inizio conversazione) ===
Sei un assistente di supporto clienti di primo livello per un negozio online di giardinaggio.
La data di oggi è 2026-10-04: usala per interpretare "oggi", "domani" e le scadenze.

Strumenti:
- search_catalog: prodotti, prezzi, coltivazione, spedizioni, resi, garanzie, regole di escalation.
- get_order_status: stato di un ordine. Passa l'ID ESATTAMENTE come lo scrive il cliente (formato GTPO seguito da 4 cifre). Se il cliente parla di un ordine senza darne l'ID: se in "Ordini citati finora" c'è un solo ordine usa quello; se ce ne sono più di uno o nessuno, chiedi di quale si tratta.
- escalate_to_human: passaggio a un operatore umano.

Regole:
1. Se la domanda ha più parti, usa i tool necessari per OGNI parte prima di rispondere.
2. Rispondi solo con informazioni ott

## Step 4.3 - Loop ReAct

In [42]:
FORCED_ANSWER_INSTRUCTION = (
    "Non puoi più usare strumenti. Rispondi ora al cliente usando SOLO le "
    "informazioni già raccolte qui sopra; se qualcosa manca, dichiaralo."
)
FALLBACK_ANSWER = (
    "Mi scuso, non riesco a completare la risposta. Ti consiglio di contattare "
    "un operatore del supporto."
)

In [43]:
def execute_tool_call(call: dict) -> ToolMessage:
    """Run one tool call requested by the model and wrap the result in a ToolMessage.

    Errors (unknown tool, invalid arguments, exceptions) are returned as text,
    so the model can see them and correct itself instead of crashing the loop.
    """
    tool_fn = TOOLS_BY_NAME.get(call["name"])
    if tool_fn is None:
        content = f"Errore: il tool '{call['name']}' non esiste."
    else:
        try:
            content = tool_fn.invoke(call["args"])
        except Exception as exc:
            content = f"Errore nell'esecuzione del tool: {str(exc)[:300]}"
    return ToolMessage(content=content, tool_call_id=call["id"], name=call["name"])

In [44]:
def force_final_answer(
    history: list[BaseMessage],
    new_messages: list[BaseMessage],
    system_prompt: str,
) -> AIMessage:
    """Force a final answer without tools, once the iteration limit is reached.

    The history is flattened into plain text, so the model has no tool-call
    pattern to continue. If the call fails, a safe fallback message is returned.
    """
    observations = "\n\n".join(
        f"Risultato {m.name}:\n{content_to_text(m.content)}"
        for m in new_messages
        if isinstance(m, ToolMessage)
    )
    prompt = (
        f"CONVERSAZIONE:\n{format_transcript(history)}\n\n"
        f"INFORMAZIONI RACCOLTE IN QUESTO TURNO:\n{observations or 'nessuna'}\n\n"
        f"{FORCED_ANSWER_INSTRUCTION}"
    )
    try:
        return call_with_backoff(
            lambda: llm.invoke([SystemMessage(system_prompt), HumanMessage(prompt)])
        )
    except Exception as exc:
        print(f"⚠️ Risposta forzata non riuscita ({type(exc).__name__}): uso il fallback")
        return AIMessage(content=FALLBACK_ANSWER)

In [45]:
def run_react_loop(
    history: list[BaseMessage],
    system_prompt: str,
    max_iterations: int = MAX_ITERATIONS,
    verbose: bool = True,
) -> list[BaseMessage]:
    """Run the ReAct loop (reason -> act -> observe) and return the new messages.

    The loop ends when the model answers without tool calls. If max_iterations
    is reached, a final call without tools forces the model to answer.
    Rate-limit errors are retried with the wait suggested by the provider.
    The input history is not modified: the caller decides what to keep.
    """
    new_messages: list[BaseMessage] = []

    for step in range(1, max_iterations + 1):
        ai_message = call_with_backoff(
            lambda: llm_with_tools.invoke(
                [SystemMessage(system_prompt)] + history + new_messages
            )
        )
        new_messages.append(ai_message)

        if not ai_message.tool_calls:
            if verbose:
                print(f"  [passo {step}] risposta finale")
            return new_messages

        for call in ai_message.tool_calls:
            result = execute_tool_call(call)
            new_messages.append(result)
            if verbose:
                args = json.dumps(call["args"], ensure_ascii=False)
                preview = str(result.content).replace("\n", " ")[:80]
                print(f"  [passo {step}] {call['name']}({args}) -> {preview}...")

    if verbose:
        print(f"  ⚠️ limite di {max_iterations} passaggi raggiunto: risposta forzata")
    new_messages.append(force_final_answer(history, new_messages, system_prompt))
    return new_messages

### Test

In [46]:
# Test 1: domanda doppia della traccia (due tool)
history = [HumanMessage("Quando arriva l'ordine GTPO1042? Posso piantare adesso i bulbi di tulipano?")]
prompt = build_system_prompt("", ["GTPO1042"])
print("=== TEST 1: domanda doppia ===")
msgs = run_react_loop(history, prompt)
called = {c["name"] for m in msgs if isinstance(m, AIMessage) for c in m.tool_calls}
final_msg = msgs[-1]
print(f"\nRISPOSTA: {final_msg.content}")
assert {"get_order_status", "search_catalog"} <= called, f"Tool chiamati: {called}"
assert isinstance(final_msg, AIMessage) and not final_msg.tool_calls and final_msg.content
print("✅ Test 1 superato")

# Test 2: nessun tool
print("\n=== TEST 2: saluto ===")
msgs = run_react_loop([HumanMessage("Ciao!")], build_system_prompt("", []))
assert len(msgs) == 1 and not msgs[0].tool_calls
print(f"RISPOSTA: {msgs[0].content}")
print("✅ Test 2 superato")

# Test 3: tool inesistente e argomenti non validi
print("\n=== TEST 3: errori dei tool ===")
err1 = execute_tool_call({"name": "inesistente", "args": {}, "id": "x1"})
err2 = execute_tool_call({"name": "get_order_status", "args": {}, "id": "x2"})
print(err1.content)
print(err2.content)
assert err1.content.startswith("Errore") and err2.content.startswith("Errore")
print("✅ Test 3 superato")

# Test 4: limite di iterazioni
print("\n=== TEST 4: limite a 1 passaggio ===")
msgs = run_react_loop(history, prompt, max_iterations=1)
assert isinstance(msgs[-1], AIMessage) and not msgs[-1].tool_calls and msgs[-1].content
print(f"RISPOSTA FORZATA: {msgs[-1].content}")
print("✅ Test 4 superato")

=== TEST 1: domanda doppia ===
  [passo 1] get_order_status({"order_id": "GTPO1042"}) -> Ordine GTPO1042 Stato: spedito Data ordine: 2026-10-02 Consegna stimata: 2026-10...
  [passo 2] search_catalog({"query": "piantare bulbi di tulipano quando"}) -> [source: guides/bulbi_autunnali.md | score: 0.74] Guida ai bulbi autunnali (tuli...
  [passo 3] risposta finale

RISPOSTA: L’ordine **GTPO1042** è stato spedito ed è in consegna. La data di consegna stimata è **2026‑10‑05**, quindi dovrebbe arrivare **domani**.

Per quanto riguarda i bulbi di tulipano, la guida indica che i bulbi a fioritura primaverile (come i tulipani) vanno piantati in autunno, da **ottobre a novembre**, quando la temperatura del suolo scende sotto i 10 °C. Oggi è il 4 ottobre, quindi è il periodo ideale per piantarli subito.  

Buona piantagione e buona attesa dei fiori! 🌷
✅ Test 1 superato

=== TEST 2: saluto ===

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debu

## Step 4.4 - Output strutturato

In [47]:
SOURCE_PATTERN = re.compile(r"\[source:\s*([^\s|\]]+)")
PENDING_MARKER = "in attesa di operatore"

In [48]:
class ResponseAssessment(BaseModel):
    """Model-generated assessment of the final answer (not the answer itself)."""

    confidence: Literal["low", "medium", "high"] = Field(
        description=(
            "'high' se la risposta è interamente supportata dai risultati dei tool; "
            "'medium' se solo in parte; 'low' se le informazioni mancano, non sono "
            "state trovate o la risposta è incerta."
        )
    )
    needs_human: bool = Field(
        description="True se il cliente deve essere preso in carico da un operatore umano."
    )
    used_sources: list[str] = Field(
        default_factory=list,
        description=(
            "File 'source' (come compaiono nei risultati di search_catalog) "
            "effettivamente usati per scrivere la risposta. Vuoto se nessuno."
        ),
    )

In [49]:
ASSESSMENT_PROMPT = """Valuta la risposta di un assistente di supporto clienti.

DOMANDA DEL CLIENTE:
{question}

RISULTATI DEI TOOL IN QUESTO TURNO:
{observations}

RISPOSTA DELL'ASSISTENTE:
{answer}

Compila la valutazione. Per used_sources indica solo i file che hanno davvero
contribuito alla risposta, ignorando risultati non pertinenti.

Rispondi SOLO con un oggetto JSON con esattamente questi campi:
{{"confidence": "low" | "medium" | "high", "needs_human": true | false, "used_sources": ["file1", "file2"]}}"""

In [50]:
class JsonAssessor:
    """Fills ResponseAssessment using JSON mode, validated with Pydantic.

    Uses a separate model from the agent, so it does not consume the agent's
    token quota. Exposes .invoke(prompt) like a LangChain runnable, so
    build_agent_response does not need to change.
    """

    MAX_ATTEMPTS = 3

    def invoke(self, prompt: str) -> ResponseAssessment:
        """Call the model in JSON mode and validate the result against the schema.

        Retries when the model returns invalid JSON (a transient failure of
        reasoning models); the fallback in build_agent_response is used only
        after MAX_ATTEMPTS failures.
        """
        last_error = None
        for _ in range(self.MAX_ATTEMPTS):
            try:
                response = call_with_backoff(
                    lambda: litellm.completion(
                        model=ASSESSOR_MODEL,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0,
                        max_tokens=2000,  # i modelli di ragionamento consumano token pensando
                        response_format={"type": "json_object"},
                    )
                )
                return ResponseAssessment.model_validate_json(response.choices[0].message.content)
            except (litellm.BadRequestError, ValidationError) as exc:
                last_error = exc
        raise last_error

assessor = JsonAssessor()

In [51]:
def extract_sources(new_messages: list[BaseMessage]) -> list[str]:
    """Return the knowledge-base files returned by search_catalog in this turn."""
    found = []
    for msg in new_messages:
        if isinstance(msg, ToolMessage) and msg.name == "search_catalog":
            found.extend(SOURCE_PATTERN.findall(content_to_text(msg.content)))
    return list(dict.fromkeys(found))

In [52]:
def escalation_ticket_created(new_messages: list[BaseMessage]) -> bool:
    """True if escalate_to_human actually created a ticket in this turn."""
    return any(
        isinstance(msg, ToolMessage)
        and msg.name == "escalate_to_human"
        and content_to_text(msg.content).startswith("Segnalazione creata")
        for msg in new_messages
    )

In [53]:
def case_pending_with_operator(new_messages: list[BaseMessage]) -> bool:
    """True if an order looked up in this turn already has a case waiting for an operator."""
    return any(
        isinstance(msg, ToolMessage)
        and msg.name == "get_order_status"
        and PENDING_MARKER in content_to_text(msg.content).lower()
        for msg in new_messages
    )

In [78]:
def build_agent_response(
    history: list[BaseMessage], new_messages: list[BaseMessage]
) -> AgentResponse:
    """Build the structured response, enforcing sources and needs_human in code.

    - answer: the final text of the ReAct loop, not rewritten.
    - sources: only files actually returned by search_catalog and used by the model.
    - needs_human: forced to True if a ticket was created in this conversation or if the
      order already has a case waiting for an operator.
    - if the assessment call fails: confidence 'low' and needs_human True.
    """
    answer = content_to_text(new_messages[-1].content).strip()
    question = next(
        (content_to_text(m.content) for m in reversed(history) if isinstance(m, HumanMessage)),
        "",
    )

    retrieved = extract_sources(new_messages)

    current_observations = [
        f"- {m.name}: {content_to_text(m.content)[:400]}"
        for m in new_messages
        if isinstance(m, ToolMessage)
    ]
    if retrieved:
        current_observations.append(f"- Fonti recuperate in questo turno: {', '.join(retrieved)}")

    past_observations = [
        f"- (turno precedente) {m.name}: {content_to_text(m.content)[:400]}"
        for m in history
        if isinstance(m, ToolMessage)
    ]
    observations = "\n".join(past_observations + current_observations) or "nessun tool usato"

    forced_human = (
    escalation_ticket_created(new_messages)
    or escalation_ticket_created(history)  # ticket già aperto in un turno precedente
    or case_pending_with_operator(new_messages)
    or case_pending_with_operator(history)  # ordine già in una pratica in un turno precedente
    )

    try:
        assessment = assessor.invoke(
            ASSESSMENT_PROMPT.format(question=question, observations=observations, answer=answer)
        )
        confidence, needs_human = assessment.confidence, assessment.needs_human
        sources = [s for s in assessment.used_sources if s in retrieved]
    except Exception as exc:
        print(f"⚠️ Valutazione non riuscita ({type(exc).__name__}): {str(exc)[:200]}")
        confidence, needs_human, sources = "low", True, []

    return AgentResponse(
        answer=answer,
        confidence=confidence,
        sources=sources,
        needs_human=needs_human or forced_human,
    )

### Test

In [55]:
# --- Test deterministici (nessuna chiamata al modello) ---
fake_new = [
    ToolMessage("[source: guides/a.md | score: 0.80]\ntesto\n\n[source: policies/b.md | score: 0.60]\ntesto",
                tool_call_id="1", name="search_catalog"),
    ToolMessage("Errore nell'esecuzione del tool: boom", tool_call_id="2", name="escalate_to_human"),
    AIMessage("ok"),
]
assert extract_sources(fake_new) == ["guides/a.md", "policies/b.md"]
assert escalation_ticket_created(fake_new) is False
fake_new[1] = ToolMessage("Segnalazione creata. Ticket: abc12345 (priorità high).", tool_call_id="2", name="escalate_to_human")
assert escalation_ticket_created(fake_new) is True
print("✅ A. extract_sources e escalation_ticket_created")


def ask(question: str) -> AgentResponse:
    """Run one stateless turn: ReAct loop + structured response."""
    time.sleep(20)  # pausa per non superare il limite di token al minuto
    history = [HumanMessage(question)]
    new = run_react_loop(history, build_system_prompt("", extract_order_ids(question)), verbose=False)
    return build_agent_response(history, new)


# --- Test con il modello ---
r1 = ask("Quando arriva l'ordine GTPO1042? Posso piantare adesso i bulbi di tulipano?")
print(r1.model_dump_json(indent=2))
assert r1.needs_human is False and r1.sources == ["guides/bulbi_autunnali.md"]
print("✅ B. domanda doppia")

r2 = ask("Il mio ordine GTPO1046 è arrivato con due confezioni danneggiate.")
print(r2.model_dump_json(indent=2))
assert r2.needs_human is True
print("✅ C. escalation: needs_human forzato a True")

r3 = ask("Come si pota un bonsai?")
print(r3.model_dump_json(indent=2))
assert r3.sources == [] and r3.confidence != "high"
print("✅ D. domanda senza risposta nella KB")

r4 = ask("Ciao!")
assert r4.sources == [] and r4.needs_human is False
print("✅ E. saluto")

# --- Fallback se la valutazione fallisce ---
class BrokenAssessor:
    """Test double that always raises."""
    def invoke(self, *args, **kwargs):
        raise RuntimeError("simulato")

working_assessor, assessor = assessor, BrokenAssessor()
try:
    r5 = ask("Ciao!")
finally:
    assessor = working_assessor
assert r5.confidence == "low" and r5.needs_human is True and r5.sources == []
print("✅ F. fallback prudente")

✅ A. extract_sources e escalation_ticket_created

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

⏳ Rate limit: attendo 10s (tentativo 1/4)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

⏳ Rate limit: attendo 30s (tentativo 2/4)
{
  "answer": "L’ordine **GTPO1042** è stato spedito ed è previsto per la consegna **domani, 5 ottobre 2026** (tracking BRT1042IT).\n\nI bulbi di tulipano Darwin Ibrido si piantano in autunno, da **ottobre a novembre**, quando la temperatura del suolo scende sotto i 10 °C. Essendo oggi **4 ottobre 2026**, è il periodo ideale per piantarli subito (a circa 15 cm di profondità, con il lato appuntito rivolto verso l’alto).  \n\nBuona piantagione e buona fioritura! 🌷",
  "confidence": "high",
  "sources": [
    "guides/bulbi_autunnali.md"
  ],
  "needs_human":

## Step 4.5 - Funzione di turno con memoria

In [79]:
@dataclass
class ConversationState:
    """Short-term memory of one conversation (one instance per session)."""

    messages: list[BaseMessage] = field(default_factory=list)
    summary: str = ""
    order_ids: list[str] = field(default_factory=list)


def chat_turn(state: ConversationState, user_message: str, verbose: bool = True) -> AgentResponse:
    """Run one conversation turn: memory management, ReAct loop and structured response.

    The state is updated in place: the new messages (including tool calls and
    results) are appended to the history after the response is built.
    """
    state.messages.append(HumanMessage(user_message))
    state.messages, state.summary, state.order_ids = manage_memory(
        state.messages, state.summary, state.order_ids
    )
    system_prompt = build_system_prompt(state.summary, state.order_ids)
    new_messages = run_react_loop(state.messages, system_prompt, verbose=verbose)
    response = build_agent_response(state.messages, new_messages)
    state.messages.extend(new_messages)
    return response

### Test

In [82]:
state = ConversationState()
conversation = [
    "Ciao, dov'è il mio ordine GTPO1042?",
    "Quando si piantano i tulipani?",
    "Quanto costa la spedizione?",
    "Come si pota la lavanda?",
    "Scusa, e il mio ordine quando arriva?",   # ID implicito: deve usare GTPO1042
]

for i, message in enumerate(conversation, 1):
    time.sleep(20)
    print(f"\n=== TURNO {i}: {message}")
    response = chat_turn(state, message)
    print(f"RISPOSTA: {response.answer}")
    print(f"[confidence={response.confidence} | sources={response.sources} | needs_human={response.needs_human}]")
    print(f"Turno {i}: messaggi={len(state.messages)}, "
          f"riassunto={'sì' if state.summary else 'no'}, ordini={state.order_ids}")

print("\n=== RIASSUNTO FINALE ===")
print(state.summary)

# Controlli
last_human = max(i for i, m in enumerate(state.messages) if isinstance(m, HumanMessage))
last_turn_tools = [
    (m.name, content_to_text(m.content))
    for m in state.messages[last_human:]
    if isinstance(m, ToolMessage)
]
assert state.order_ids == ["GTPO1042"]
assert state.summary, "La summarization non è mai scattata"
assert not has_orphan_tool_messages(state.messages)
assert any(name == "get_order_status" and "GTPO1042" in text for name, text in last_turn_tools), \
    f"Turno 5: ordine non consultato. Tool: {last_turn_tools}"
print("\n✅ Memoria: ID implicito risolto dopo il riassunto")


=== TURNO 1: Ciao, dov'è il mio ordine GTPO1042?
  [passo 1] get_order_status({"order_id": "GTPO1042"}) -> Ordine GTPO1042 Stato: spedito Data ordine: 2026-10-02 Consegna stimata: 2026-10...
  [passo 2] risposta finale
RISPOSTA: Buongiorno!  

Il suo ordine **GTPO1042** è stato spedito.  
- **Data ordine:** 2 ottobre 2026  
- **Consegna stimata:** 5 ottobre 2026 (domani)  
- **Tracking:** BRT1042IT  

Al momento il pacco è in transito e dovrebbe arrivare domani. Se avesse bisogno di ulteriori informazioni, sono a disposizione. 🌱
[confidence=high | sources=[] | needs_human=False]
Turno 1: messaggi=4, riassunto=no, ordini=['GTPO1042']

=== TURNO 2: Quando si piantano i tulipani?
  [passo 1] search_catalog({"query": "quando piantare i tulipani"}) -> [source: guides/bulbi_autunnali.md | score: 0.72] Guida ai bulbi autunnali (tuli...
  [passo 2] risposta finale
RISPOSTA: I bulbi a fioritura primaverile, come i **tulipani**, vanno piantati in **autunno**, da **ottobre a novembre**, quando l

# Step 5 - Valutazione (RAGAS - Deepeval)


## Relazione Step 5

**Dataset.** 18 conversazioni (23 turni) in 7 categorie: kb (6), order (2), multi_tool (2), escalation (3), not_found (1), out_of_scope (2), multi_turn (2). Ogni caso ha una risposta di riferimento e le aspettative su tool, fonti, `needs_human` e `confidence`. Si valuta solo l'ultimo turno di ogni conversazione.

**Controlli deterministici.** Per ogni caso si verificano tool chiamati, fonti, `needs_human` e `confidence`. Il tasso di successo è stato 18/18 sulla versione v1 finale e 17/18 su due run complete della v2. Il caso che fallisce cambia da una run all'altra (mem_02, poi oos_02): il sistema non è deterministico e un singolo punteggio su 18 casi ha un margine di circa ±1 caso.

**Metriche con giudice LLM.** Giudice: `gpt-4.1-mini` (OpenAI), di una famiglia diversa dall'agente (`gpt-oss-120b` su Groq) per ridurre il bias di auto-preferenza.
- RAGAS: *Faithfulness* (risposta rispetto a tutti gli output dei tool) e *Context Precision* (contesti recuperati rispetto alla risposta di riferimento).
- DeepEval: *GEval* di correttezza (soglia 0.7) e *Answer Relevancy*.

| Metrica | v1 | v2 |
|---|---|---|
| Faithfulness (17 casi con contesto) | 0.61 | 0.68 |
| Faithfulness (kb, order, multi_tool) | 0.75 | 0.78 |
| Context precision (9 casi) | 0.89 | 0.89 |
| Correctness | 0.74 | 0.81 |
| Answer relevancy | 0.84 | 0.85 |

**Iterazione.** Dalla v1 sono emersi due difetti reali: dettagli aggiunti oltre le fonti (kb_05, kb_02) e risposte di escalation generiche, senza ordine né motivo (esc_01, esc_02). Nella v2 si sono rafforzate la regola 2 (solo informazioni dei tool) e si è aggiunta la regola 7 (citare ordine e motivo, dire se la pratica è già aperta), più una guardia nel codice che mantiene `needs_human` anche per pratiche già in carico in turni precedenti. Effetto misurato: la correttezza delle escalation sale da 0.40 a 0.77; la regola 2 non ha un effetto netto (kb_05 resta a 0.0 di faithfulness).

**Limiti.**
- Non determinismo: l'esito cambia tra run uguali; differenze sotto 0.1 non sono significative.
- Dataset piccolo: le regole del prompt sono state ritoccate dopo aver visto i risultati (rischio di overfitting sui 18 casi).
- Faithfulness poco interpretabile fuori dai casi fattuali: penalizza formule di cortesia e numeri di ticket (escalation, out_of_scope, not_found, multi_turn).
- Answer relevancy penalizza i rifiuti corretti (oos_01: 0.0 con correctness 0.9).
- Il giudice varia: la stessa risposta può ricevere faithfulness molto diverse da giudici diversi (esc_01: 0.0 con Groq, 0.67 con OpenAI).
- Debolezza residua: nei follow-up (mem_02) l'agente non richiama i tool e risponde in modo vago; oos_02 effettua 3–5 ricerche e a volte assegna `confidence` alta a una risposta "non so".

**Possibili sviluppi.** Definire esplicitamente i livelli di `confidence` nel prompt dell'assessor; ripetere le run (3+) e riportare media e deviazione; usare un giudice diverso per confronto incrociato.

## Step 5.1 - Caricamento dataset e costanti

In [58]:
EVAL_DATASET_PATH = "data/eval_dataset.json"
EVAL_RESULTS_PATH = "data/eval_runs.json"

# Pause per restare sotto il limite di 8000 TPM di Groq
PAUSE_BETWEEN_TURNS = 8
PAUSE_BETWEEN_CASES = 15

with open(EVAL_DATASET_PATH, encoding="utf-8") as f:
    EVAL_CASES = json.load(f)

print(f"Casi caricati: {len(EVAL_CASES)}")

Casi caricati: 18


## Step 5.2 - Runner e controlli deterministici

In [59]:
def extract_last_turn_trace(messages: list) -> tuple[list[str], list[str], list[str]]:
    """Estrae dall'ultimo turno i tool chiamati e i testi restituiti dai tool.

    Returns:
        tools_called: nomi dei tool chiamati, in ordine.
        contexts_kb: chunk restituiti da search_catalog (per le metriche di retrieval).
        contexts_all: output di tutti i tool (per la faithfulness).
    """
    last_human_idx = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
    turn_messages = messages[last_human_idx + 1:]

    tool_names_by_id: dict[str, str] = {}
    tools_called: list[str] = []
    for message in turn_messages:
        if isinstance(message, AIMessage):
            for call in message.tool_calls or []:
                tool_names_by_id[call["id"]] = call["name"]
                tools_called.append(call["name"])

    contexts_kb: list[str] = []
    contexts_all: list[str] = []
    for message in turn_messages:
        if isinstance(message, ToolMessage):
            text = str(message.content)
            contexts_all.append(text)
            if tool_names_by_id.get(message.tool_call_id) == "search_catalog":
                chunks = re.split(r"(?=\[source:)", text)
                contexts_kb.extend(c.strip() for c in chunks if c.strip().startswith("[source:"))

    return tools_called, contexts_kb, contexts_all

In [60]:
def compute_programmatic_checks(case: dict, record: dict) -> dict:
    """Confronta il comportamento osservato con quello atteso.

    Ogni controllo vale True/False, oppure None se nel caso non è valutato.
    """
    called = set(record["tools_called"])
    checks: dict[str, bool | None] = {}

    checks["tools_ok"] = (
        set(case["expected_tools"]) <= called
        and not (set(case["forbidden_tools"]) & called)
    )

    expected_sources = case["expected_sources"]
    if expected_sources is None:
        checks["sources_ok"] = None
    elif expected_sources == []:
        checks["sources_ok"] = len(record["sources"]) == 0
    else:
        checks["sources_ok"] = bool(set(expected_sources) & set(record["sources"]))

    expected_needs_human = case["expected_needs_human"]
    checks["needs_human_ok"] = (
        None if expected_needs_human is None
        else record["needs_human"] == expected_needs_human
    )

    expected_confidence = case["expected_confidence"]
    checks["confidence_ok"] = (
        None if expected_confidence is None
        else record["confidence"] in expected_confidence
    )

    applicable = [v for v in checks.values() if v is not None]
    checks["passed"] = all(applicable)
    return checks

In [61]:
def run_case(case: dict) -> dict:
    """Esegue un caso su una conversazione nuova e ne registra l'esito."""
    state = ConversationState()
    turns = case["turns"]
    record = {
        "id": case["id"],
        "category": case["category"],
        "question": turns[-1],
        "reference_answer": case["reference_answer"],
        "error": None,
    }

    try:
        for i, user_message in enumerate(turns):
            start = time.time()
            response = chat_turn(state, user_message, verbose=False)
            latency = time.time() - start
            if i < len(turns) - 1:
                time.sleep(PAUSE_BETWEEN_TURNS)

        tools_called, contexts_kb, contexts_all = extract_last_turn_trace(state.messages)
        record.update({
            "answer": response.answer,
            "confidence": response.confidence,
            "sources": list(response.sources),
            "needs_human": response.needs_human,
            "tools_called": tools_called,
            "contexts_kb": contexts_kb,
            "contexts_all": contexts_all,
            "latency_s": round(latency, 1),
        })
        record.update(compute_programmatic_checks(case, record))
    except Exception as exc:  # un errore su un caso non deve fermare tutta la run
        record.update({
            "answer": "", "confidence": None, "sources": [], "needs_human": None,
            "tools_called": [], "contexts_kb": [], "contexts_all": [], "latency_s": None,
            "tools_ok": False, "sources_ok": False, "needs_human_ok": False,
            "confidence_ok": False, "passed": False,
            "error": f"{type(exc).__name__}: {exc}",
        })
    return record

In [62]:
def run_evaluation(
    cases: list[dict],
    results_path: str = EVAL_RESULTS_PATH,
    only_ids: list[str] | None = None,
    force: bool = False,
) -> list[dict]:
    """Esegue i casi e salva i risultati dopo ognuno, così la run è riprendibile.

    I casi già presenti nel file (senza errori) vengono saltati, a meno di force=True.
    """
    done: dict[str, dict] = {}
    if os.path.exists(results_path) and not force:
        with open(results_path, encoding="utf-8") as f:
            done = {r["id"]: r for r in json.load(f)}

    for case in cases:
        if only_ids and case["id"] not in only_ids:
            continue
        if case["id"] in done and not done[case["id"]]["error"]:
            print(f"= {case['id']}: già eseguito, salto")
            continue

        print(f"> {case['id']} ({case['category']}, {len(case['turns'])} turni)")
        done[case["id"]] = run_case(case)
        status = "OK" if done[case["id"]]["passed"] else "KO"
        print(f"  {status} | tools={done[case['id']]['tools_called']}")

        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(list(done.values()), f, ensure_ascii=False, indent=2)
        time.sleep(PAUSE_BETWEEN_CASES)

    return [done[c["id"]] for c in cases if c["id"] in done]

## Step 5.3 - Esecuzione e riepilogo

In [95]:
def results_to_dataframe(results: list[dict]) -> pd.DataFrame:
    """Tabella di riepilogo con i controlli deterministici."""
    rows = []
    for r in results:
        rows.append({
            "id": r["id"],
            "category": r["category"],
            "tools_ok": r["tools_ok"],
            "sources_ok": r["sources_ok"],
            "needs_human_ok": r["needs_human_ok"],
            "confidence_ok": r["confidence_ok"],
            "passed": r["passed"],
            "tools_called": ", ".join(r["tools_called"]),
            "sources": ", ".join(r["sources"]),
            "confidence": r["confidence"],
            "needs_human": r["needs_human"],
            "latency_s": r["latency_s"],
            "error": r["error"],
        })
    return pd.DataFrame(rows)

In [96]:
results = run_evaluation(EVAL_CASES, force=False)

df_checks = results_to_dataframe(results)
display(df_checks)

check_cols = ["tools_ok", "sources_ok", "needs_human_ok", "confidence_ok", "passed"]
print("\nAccuratezza per controllo (casi in cui è applicabile):")
print(df_checks[check_cols].apply(lambda s: s.dropna().astype(bool).mean()).round(2))

print("\nTasso di successo per categoria:")
print(df_checks.groupby("category")["passed"].apply(lambda s: s.astype(bool).mean()).round(2))

print("\nCasi falliti:")
for r in results:
    if not r["passed"]:
        failed = [c for c in check_cols[:-1] if r[c] is False]
        print(f"- {r['id']}: {failed} | {r['error'] or r['answer'][:200]}")

= kb_01: già eseguito, salto
= kb_02: già eseguito, salto
= kb_03: già eseguito, salto
= kb_04: già eseguito, salto
= kb_05: già eseguito, salto
= kb_06: già eseguito, salto
= ord_01: già eseguito, salto
= ord_02: già eseguito, salto
= mt_01: già eseguito, salto
= mt_02: già eseguito, salto
= esc_01: già eseguito, salto
= esc_02: già eseguito, salto
= esc_03: già eseguito, salto
= nf_01: già eseguito, salto
= oos_01: già eseguito, salto
= oos_02: già eseguito, salto
= mem_01: già eseguito, salto
= mem_02: già eseguito, salto


,id,category,tools_ok,sources_ok,needs_human_ok,confidence_ok,passed,tools_called,sources,confidence,needs_human,latency_s,error
0,kb_01,kb,True,True,True,True,True,search_catalog,policies/spedizioni.md,high,False,2.0,None
1,kb_02,kb,True,True,True,True,True,"search_catalog, search_catalog",policies/resi.md,high,False,3.2,None
2,kb_03,kb,True,True,True,True,True,search_catalog,products/olivo_leccino.md,high,False,1.3,None
3,kb_04,kb,True,True,True,True,True,search_catalog,guides/cura_lavanda.md,high,False,1.6,None
4,kb_05,kb,True,True,True,True,True,search_catalog,guides/parassiti_comuni.md,high,False,1.6,None
5,kb_06,kb,True,True,True,True,True,search_catalog,products/kit_irrigazione_goccia.md,high,False,1.8,None
6,ord_01,order,True,True,True,True,True,get_order_status,,high,False,1.3,None
7,ord_02,order,True,True,True,True,True,get_order_status,,high,False,1.5,None
8,mt_01,multi_tool,True,True,True,True,True,"get_order_status, search_catalog",guides/bulbi_autunnali.md,high,False,3.5,None
9,mt_02,multi_tool,True,True,True,True,True,"get_order_status, search_catalog",products/rosa_gertrude_jekyll.md,high,False,4.1,None



Accuratezza per controllo (casi in cui è applicabile):
tools_ok          1.00
sources_ok        1.00
needs_human_ok    1.00
confidence_ok     0.94
passed            0.94
dtype: float64

Tasso di successo per categoria:
category
escalation      1.0
kb              1.0
multi_tool      1.0
multi_turn      1.0
not_found       1.0
order           1.0
out_of_scope    0.5
Name: passed, dtype: float64

Casi falliti:
- oos_02: ['confidence_ok'] | Al momento la nostra knowledge base contiene solo le informazioni sulla spedizione nazionale (gratuita per ordini ≥ 60 € o 5,90 € altrimenti) e non indica i costi per la spedizione in Spagna. Per cono


## Step 5.4 - Giudici e metriche

In [97]:
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"  # nessuna telemetria

from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase

try:
    from deepeval.test_case import SingleTurnParams as TestCaseParams
except ImportError:  # versioni più vecchie di DeepEval
    from deepeval.test_case import LLMTestCaseParams as TestCaseParams

from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, Faithfulness

# Giudice: lo stesso modello dell'agente, ma l'agente non gira in questa fase,
# quindi tutto il budget di token al minuto è per la valutazione
JUDGE_MODEL = "openai/gpt-4.1-mini" # prima: "groq/openai/gpt-oss-120b"
JUDGE_MAX_TOKENS = 4000  # i modelli di ragionamento consumano token pensando
PAUSE_BETWEEN_SAMPLES = 10

RAGAS_RESULTS_PATH = "data/eval_ragas.json"
DEEPEVAL_RESULTS_PATH = "data/eval_deepeval.json"

In [98]:
def load_runs(path: str = EVAL_RESULTS_PATH) -> list[dict]:
    """Load the conversations executed by the evaluation runner."""
    with open(path, encoding="utf-8") as f:
        return json.load(f)

async def acompletion_with_backoff(*args, max_attempts: int = 6, **kwargs):
    """litellm.acompletion that waits the time suggested by Groq on rate-limit errors."""
    for attempt in range(1, max_attempts + 1):
        try:
            return await litellm.acompletion(*args, **kwargs)
        except litellm.RateLimitError as exc:
            if attempt == max_attempts:
                raise
            match = RETRY_WAIT_PATTERN.search(str(exc))
            wait = float(match.group(1)) + 1 if match else 15 * attempt
            print(f"⏳ Rate limit (giudice): attendo {wait:.0f}s (tentativo {attempt}/{max_attempts})")
            await asyncio.sleep(wait)

# --- Giudice per RAGAS: client LiteLLM asincrono in JSON mode (via instructor) ---
ragas_llm = llm_factory(
    JUDGE_MODEL,
    client=instructor.from_litellm(acompletion_with_backoff, mode=instructor.Mode.JSON),
    adapter="litellm",
    temperature=0,
    max_tokens=JUDGE_MAX_TOKENS,
    num_retries=0, # i tentativi li gestisce acompletion_with_backoff
)
faithfulness_metric = Faithfulness(llm=ragas_llm)
context_precision_metric = ContextPrecision(llm=ragas_llm)


# --- Giudice per DeepEval: stessa tecnica del JsonAssessor (JSON mode + Pydantic) ---
class GroqJudge(DeepEvalBaseLLM):
    """Judge model for DeepEval: Groq through LiteLLM, JSON mode validated with Pydantic."""

    MAX_ATTEMPTS = 3

    def __init__(self, model: str = JUDGE_MODEL):
        self.model_name = model

    def load_model(self):
        return self.model_name

    def get_model_name(self) -> str:
        return self.model_name

    def generate(self, prompt: str, schema: type[BaseModel] | None = None):
        """Return plain text, or an instance of `schema` when DeepEval asks for structure."""
        messages = [{"role": "user", "content": prompt}]
        extra = {}
        if schema is not None:
            instruction = (
                "Rispondi SOLO con un oggetto JSON valido conforme a questo JSON Schema:\n"
                + json.dumps(schema.model_json_schema())
            )
            messages.insert(0, {"role": "system", "content": instruction})
            extra["response_format"] = {"type": "json_object"}

        last_error = None
        for _ in range(self.MAX_ATTEMPTS):
            try:
                response = call_with_backoff(
                    lambda: litellm.completion(
                        model=self.model_name,
                        messages=messages,
                        temperature=0,
                        max_tokens=JUDGE_MAX_TOKENS,
                        **extra,
                    )
                )
                text = response.choices[0].message.content
                return schema.model_validate_json(text) if schema is not None else text
            except (litellm.BadRequestError, ValidationError) as exc:
                last_error = exc
        raise last_error

    async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None):
        return self.generate(prompt, schema)


deepeval_judge = GroqJudge()

correctness_metric = GEval(
    name="Correttezza",
    criteria=(
        "Valuta se la risposta reale è corretta rispetto alla risposta attesa: stessi fatti, "
        "numeri, date e condizioni, senza informazioni inventate o contraddittorie. "
        "Penalizza le omissioni importanti; ignora le differenze di stile e di lunghezza."
    ),
    evaluation_params=[
        TestCaseParams.INPUT,
        TestCaseParams.ACTUAL_OUTPUT,
        TestCaseParams.EXPECTED_OUTPUT,
    ],
    model=deepeval_judge,
    threshold=0.7,
    async_mode=False,
)
answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7, model=deepeval_judge, include_reason=True, async_mode=False
)

runs = load_runs()
print(f"Conversazioni da valutare: {len(runs)}")

Conversazioni da valutare: 18


## Step 5.5 - RAGAS (faithfulness e context precision)

In [99]:
async def run_ragas(
    runs: list[dict],
    results_path: str = RAGAS_RESULTS_PATH,
    only_ids: list[str] | None = None,
    force: bool = False,
) -> dict:
    """Score faithfulness and context precision with RAGAS, saving after each case.

    - faithfulness: the answer against everything the tools returned (needs some context).
    - context_precision: only for cases where search_catalog returned chunks,
      compared with the reference answer.
    Cases without the needed context get None, not 0: the metric does not apply.
    """
    scores: dict[str, dict] = {}
    if os.path.exists(results_path) and not force:
        with open(results_path, encoding="utf-8") as f:
            scores = json.load(f)

    for run in runs:
        if only_ids and run["id"] not in only_ids:
            continue
        if run["id"] in scores and not scores[run["id"]]["error"]:
            print(f"= {run['id']}: già valutato, salto")
            continue

        entry = {"faithfulness": None, "context_precision": None, "error": None}
        try:
            if run["contexts_all"]:
                result = await faithfulness_metric.ascore(
                    user_input=run["question"],
                    response=run["answer"],
                    retrieved_contexts=run["contexts_all"],
                )
                entry["faithfulness"] = round(float(result.value), 3)
            if run["contexts_kb"]:
                result = await context_precision_metric.ascore(
                    user_input=run["question"],
                    reference=run["reference_answer"],
                    retrieved_contexts=run["contexts_kb"],
                )
                entry["context_precision"] = round(float(result.value), 3)
        except Exception as exc:  # un errore su un caso non ferma la run
            entry["error"] = f"{type(exc).__name__}: {str(exc)[:200]}"

        scores[run["id"]] = entry
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(scores, f, ensure_ascii=False, indent=2)
        print(
            f"> {run['id']}: faithfulness={entry['faithfulness']} | "
            f"context_precision={entry['context_precision']} {entry['error'] or ''}"
        )
        time.sleep(PAUSE_BETWEEN_SAMPLES)

    return scores

ragas_scores = await run_ragas(runs, force=True)

> mem_02: faithfulness=None | context_precision=None 
> kb_01: faithfulness=1.0 | context_precision=1.0 
> kb_02: faithfulness=0.909 | context_precision=1.0 
> kb_03: faithfulness=1.0 | context_precision=1.0 
> kb_04: faithfulness=1.0 | context_precision=1.0 
> kb_05: faithfulness=0.0 | context_precision=1.0 
> kb_06: faithfulness=0.667 | context_precision=1.0 
> ord_01: faithfulness=0.667 | context_precision=None 
> ord_02: faithfulness=1.0 | context_precision=None 
> mt_01: faithfulness=0.556 | context_precision=1.0 
> mt_02: faithfulness=1.0 | context_precision=0.0 
> esc_01: faithfulness=0.333 | context_precision=None 
> esc_02: faithfulness=0.667 | context_precision=None 
> esc_03: faithfulness=0.333 | context_precision=None 
> nf_01: faithfulness=0.667 | context_precision=None 
> oos_01: faithfulness=0.5 | context_precision=None 
> oos_02: faithfulness=0.667 | context_precision=1.0 
> mem_01: faithfulness=0.625 | context_precision=None 


## Step 5.6 - DeepEval (correttezza e answer relevancy)

In [100]:
def run_deepeval(
    runs: list[dict],
    results_path: str = DEEPEVAL_RESULTS_PATH,
    only_ids: list[str] | None = None,
    force: bool = False,
) -> dict:
    """Score correctness (GEval) and answer relevancy with DeepEval, saving after each case.

    - correctness: actual answer against the reference answer of the dataset.
    - answer_relevancy: does the answer address the (last) question.
    """
    scores: dict[str, dict] = {}
    if os.path.exists(results_path) and not force:
        with open(results_path, encoding="utf-8") as f:
            scores = json.load(f)

    for run in runs:
        if only_ids and run["id"] not in only_ids:
            continue
        if run["id"] in scores and not scores[run["id"]]["error"]:
            print(f"= {run['id']}: già valutato, salto")
            continue

        entry = {
            "correctness": None, "correctness_reason": None,
            "answer_relevancy": None, "answer_relevancy_reason": None,
            "error": None,
        }
        try:
            test_case = LLMTestCase(
                input=run["question"],
                actual_output=run["answer"],
                expected_output=run["reference_answer"],
            )
            correctness_metric.measure(test_case)
            entry["correctness"] = round(float(correctness_metric.score), 3)
            entry["correctness_reason"] = correctness_metric.reason

            answer_relevancy_metric.measure(test_case)
            entry["answer_relevancy"] = round(float(answer_relevancy_metric.score), 3)
            entry["answer_relevancy_reason"] = answer_relevancy_metric.reason
        except Exception as exc:  # un errore su un caso non ferma la run
            entry["error"] = f"{type(exc).__name__}: {str(exc)[:200]}"

        scores[run["id"]] = entry
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(scores, f, ensure_ascii=False, indent=2)
        print(
            f"> {run['id']}: correctness={entry['correctness']} | "
            f"answer_relevancy={entry['answer_relevancy']} {entry['error'] or ''}"
        )
        time.sleep(PAUSE_BETWEEN_SAMPLES)

    return scores


deepeval_scores = run_deepeval(runs, force=True)

Output()

Output()

> mem_02: correctness=0.3 | answer_relevancy=0.6 


Output()

Output()

...

Output()

> oos_02: correctness=0.9 | answer_relevancy=0.667 


Output()

Output()

> mem_01: correctness=0.8 | answer_relevancy=1.0 


## Step 5.7 - Tabella riassuntiva e casi deboli

In [101]:
def build_scores_table(runs: list[dict], ragas_scores: dict, deepeval_scores: dict) -> pd.DataFrame:
    """One row per conversation: deterministic check + RAGAS + DeepEval scores."""
    rows = []
    for run in runs:
        ragas_entry = ragas_scores.get(run["id"], {})
        deepeval_entry = deepeval_scores.get(run["id"], {})
        rows.append({
            "id": run["id"],
            "category": run["category"],
            "passed": run["passed"],
            "faithfulness": ragas_entry.get("faithfulness"),
            "context_precision": ragas_entry.get("context_precision"),
            "correctness": deepeval_entry.get("correctness"),
            "answer_relevancy": deepeval_entry.get("answer_relevancy"),
        })
    return pd.DataFrame(rows)


metric_cols = ["faithfulness", "context_precision", "correctness", "answer_relevancy"]
df_scores = build_scores_table(runs, ragas_scores, deepeval_scores)
display(df_scores)

print("\nMedia per metrica (solo casi in cui è applicabile):")
print(df_scores[metric_cols].apply(pd.to_numeric).mean().round(2))

KNOWLEDGE_CATEGORIES = ["kb", "order", "multi_tool"]
knowledge = df_scores[df_scores["category"].isin(KNOWLEDGE_CATEGORIES)]
print("\nFaithfulness sui soli casi di conoscenza (kb, order, multi_tool):")
print(round(pd.to_numeric(knowledge["faithfulness"]).mean(), 2))

print("\nMedia per categoria:")
print(df_scores.groupby("category")[metric_cols].apply(lambda g: g.apply(pd.to_numeric).mean()).round(2))

THRESHOLD = 0.7
print(f"\nCasi con almeno una metrica sotto {THRESHOLD}:")

for _, row in df_scores.iterrows():
    low = {m: row[m] for m in metric_cols if pd.notna(row[m]) and row[m] < THRESHOLD}
    if low:
        scores = deepeval_scores.get(row["id"], {})
        print(f"- {row['id']}: {low}")
        if "correctness" in low:
            print(f"    correctness: {(scores.get('correctness_reason') or '')[:300]}")
        if "answer_relevancy" in low:
            print(f"    answer_relevancy: {(scores.get('answer_relevancy_reason') or '')[:300]}")
        if "faithfulness" in low or "context_precision" in low:
            print("    (RAGAS non fornisce una motivazione testuale: vedi risposta e contesti)")

,id,category,passed,faithfulness,context_precision,correctness,answer_relevancy
0,mem_02,multi_turn,True,NaN,NaN,0.3,0.600
1,kb_01,kb,True,1.000,1.0,1.0,1.000
2,kb_02,kb,True,0.909,1.0,0.9,1.000
3,kb_03,kb,True,1.000,1.0,0.7,1.000
4,kb_04,kb,True,1.000,1.0,0.9,1.000
5,kb_05,kb,True,0.000,1.0,0.9,1.000
6,kb_06,kb,True,0.667,1.0,0.9,0.750
7,ord_01,order,True,0.667,NaN,0.9,1.000
8,ord_02,order,True,1.000,NaN,0.7,1.000
9,mt_01,multi_tool,True,0.556,1.0,0.9,1.000



Media per metrica (solo casi in cui è applicabile):
faithfulness         0.68
context_precision    0.89
correctness          0.81
answer_relevancy     0.85
dtype: float64

Faithfulness sui soli casi di conoscenza (kb, order, multi_tool):
0.78

Media per categoria:
              faithfulness  context_precision  correctness  answer_relevancy
category                                                                    
escalation            0.44                NaN         0.77              0.80
kb                    0.76                1.0         0.88              0.96
multi_tool            0.78                0.5         0.85              1.00
multi_turn            0.62                NaN         0.55              0.80
not_found             0.67                NaN         0.80              0.83
order                 0.83                NaN         0.80              1.00
out_of_scope          0.58                1.0         0.90              0.33

Casi con almeno una metrica sotto 0.7:
-

In [102]:
suspects = ["kb_05", "kb_02", "esc_01", "esc_02", "mem_02", "mt_02"]
runs_by_id = {r["id"]: r for r in runs}
for case_id in suspects:
    run = runs_by_id[case_id]
    scores = deepeval_scores[case_id]
    print(f"===== {case_id} (correctness={scores['correctness']}, relevancy={scores['answer_relevancy']})")
    print("DOMANDA:   ", run["question"])
    print("RISPOSTA:  ", run["answer"])
    print("RIFERIM.:  ", run["reference_answer"])
    print("GEval:     ", scores["correctness_reason"])
    print("Relevancy: ", scores["answer_relevancy_reason"])
    print()

===== kb_05 (correctness=0.9, relevancy=1.0)
DOMANDA:    Come combatto gli afidi sulle mie rose?
RISPOSTA:   Per le rose, gli afidi si combattono con una soluzione di sapone molle al 2 % da spruzzare ogni 5 giorni.
RIFERIM.:   Spruzza una soluzione di sapone molle al 2% ogni 5 giorni.
GEval:      The Actual Output matches the Expected Output in facts and instructions, specifying a 2% soap solution sprayed every 5 days for aphids on roses. The only minor difference is the phrasing and the inclusion of 'Per le rose' which is consistent with the Input.
Relevancy:  The score is 1.00 because the response fully addresses how to combat aphids on roses without including any irrelevant information.

===== kb_02 (correctness=0.9, relevancy=1.0)
DOMANDA:    Posso restituire un paio di forbici da potatura comprate due settimane fa?
RISPOSTA:   Sì, le forbici da potatura possono essere restituite perché sono state acquistate solo due settimane fa, quindi rientrano nel limite di 30 giorni previsto d

# Step 6 - Deploy (FastAPI)

## Relazione Step 6

L'agente è esposto come API locale con FastAPI. Il codice degli Step 1-4 è stato estratto in `app/agent_core.py` (senza dipendenze Colab) e importato da `app/main.py`; le risorse pesanti (modello di embedding, vector store, client LLM) vengono create una sola volta all'avvio del server.

**Endpoint.** `POST /chat` esegue un turno e restituisce esattamente i quattro campi dell'output strutturato (`answer`, `confidence`, `sources`, `needs_human`); l'ID della conversazione viaggia nell'header `X-Session-Id` e va rimandato nel campo `session_id` per continuare la conversazione (senza `session_id` ne viene creata una nuova)

**Sessioni e memoria.** Ogni `session_id` corrisponde a un `ConversationState` (messaggi, riassunto, ordini citati) in memoria. Un lock per sessione serializza i turni della stessa conversazione; il numero di sessioni è limitato (200, scarto della meno recente).

**Robustezza.** Messaggio validato (non vuoto, massimo 1000 caratteri), errori del modello tradotti in `503` (rate limit) o `500` senza dettagli interni. Gli endpoint sono sincroni e girano nel thread pool di FastAPI, quindi le chiamate bloccanti al modello non fermano il server.

**Test.** 13 test automatici con LLM e embedding finti (nessuna chiave): sessioni isolate, memoria con riassunto, `needs_human` persistente dopo un'escalation senza secondo ticket, validazione e gestione errori.

**Limiti.** Sessioni perse al riavvio (servirebbe Redis o un database), nessuna autenticazione (il tool ordini non verifica l'identità del cliente), limiti di token del piano gratuito di Groq, primo avvio lento per il download del modello di embedding.

Istruzioni di avvio locale nel `README.md` del repository.

## Step 6.1 - Download repository e avvio del server

In [105]:
import subprocess
import sys

REPO_URL = "https://github.com/lorenzofranco24/greenthumb-support-agent.git"
REPO_DIR = "greenthumb-support-agent"
API_URL = "http://127.0.0.1:8000"

%pip install -q fastapi "uvicorn[standard]" python-dotenv

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)

# GROQ_API_KEY è già in os.environ (secret di Colab): il processo figlio la eredita
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--port", "8000"],
    cwd=REPO_DIR,
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
)

In [106]:
for _ in range(120):  # fino a 10 minuti: il primo avvio carica il modello di embedding
    try:
        health = requests.get(f"{API_URL}/health", timeout=2).json()
        break
    except requests.exceptions.ConnectionError:
        time.sleep(5)
else:
    print(open("server.log").read()[-2000:])
    raise RuntimeError("Il server non è partito")

print(health)

{'status': 'ok', 'orders': 22, 'documents': 35, 'active_sessions': 0}


## Step 6.2 - Test API

In [107]:
def ask_api(message: str, session_id: str | None = None) -> tuple[dict, str]:
    """Send one message to the API; return the JSON body and the session ID (X-Session-Id header)."""
    response = requests.post(
        f"{API_URL}/chat",
        json={"message": message, "session_id": session_id},
        timeout=120,
    )
    response.raise_for_status()
    return response.json(), response.headers["X-Session-Id"]


# 1) Domanda della traccia: servono due tool nello stesso turno (ordine + catalogo)
body, _ = ask_api("Quando arriva l'ordine GTPO1042? Posso piantare adesso i bulbi di tulipano?")
print(json.dumps(body, ensure_ascii=False, indent=2))

# 2) Conversazione su due turni: escalation e memoria
first, session_id = ask_api("Il mio ordine GTPO1046 è arrivato con due confezioni danneggiate, cosa devo fare?")
print(json.dumps(first, ensure_ascii=False, indent=2))
second, _ = ask_api("Quando mi contattate?", session_id)
print(json.dumps(second, ensure_ascii=False, indent=2))

# 3) Validazione e reset
print("Messaggio vuoto ->", requests.post(f"{API_URL}/chat", json={"message": ""}).status_code)
print("Reset sessione  ->", requests.delete(f"{API_URL}/sessions/{session_id}").status_code)

{
  "answer": "L’ordine **GTPO1042** è stato spedito ed è in consegna con data stimata **2026‑10‑05** (domani).\n\nI bulbi di tulipano Darwin Ibrido si piantano in autunno, da **ottobre a novembre**, quando la temperatura del suolo scende sotto i 10 °C. Essendo oggi **4 ottobre 2026**, è quindi il periodo ideale per piantarli subito.",
  "confidence": "high",
  "sources": [
    "guides/bulbi_autunnali.md"
  ],
  "needs_human": false
}
{
  "answer": "Gentile cliente, ho segnalato il danno da trasporto delle due confezioni del suo ordine **GTPO1046** a un operatore umano. Riceverà una risposta entro le prossime ore. Grazie per la pazienza.",
  "confidence": "high",
  "sources": [],
  "needs_human": true
}
{
  "answer": "Gli operatori prenderanno in carico la sua segnalazione entro le **10:00 del 5 ottobre 2026** (giorno feriale). Verrà contattato il prima possibile, entro le ore di apertura (lun‑ven, 9:00‑18:00).",
  "confidence": "high",
  "sources": [],
  "needs_human": true
}
Messaggi

In [108]:
# Spegne il server
server.terminate()

# Step 7 - Relazione Finale

## Obiettivo
Agente di customer support per GreenThumb, e-commerce fittizio di giardinaggio. Risponde in italiano a domande su prodotti e coltivazione, consulta lo stato degli ordini e passa a un operatore umano quando serve. Esposto come API FastAPI in locale (`POST /chat`).

## Architettura
| Componente | Scelta | Motivo |
|---|---|---|
| Agente | Loop ReAct scritto a mano (LangChain + LiteLLM, `groq/openai/gpt-oss-120b`), max 5 iterazioni | Ragionamento visibile, integrazione con la memoria, controlli deterministici attorno al modello |
| Tool | `search_catalog` (RAG), `get_order_status`, `escalate_to_human` | I tre casi d'uso della traccia |
| Retrieval | Chroma (coseno) + `BAAI/bge-m3`, 35 documenti, k=3, soglia 0.55 | Il modello iniziale MiniLM non recuperava query brevi; bge-m3 sì |
| Memoria lunga | Vector store sulla knowledge base | Conoscenza persistente interrogata per similarità |
| Memoria breve | Summary buffer (riassunto oltre 10 messaggi, finestra recente di 4) + order ID estratti con regex | Il numero d'ordine citato all'inizio non va perso |
| Output | JSON `answer`, `confidence`, `sources`, `needs_human` | `answer` dal loop; gli altri campi da una chiamata separata (`gpt-oss-20b`, JSON mode + Pydantic) |
| Controlli in codice | `sources` ⊆ fonti realmente lette; `needs_human` può solo salire | Sono i campi con conseguenze concrete |
| Deploy | FastAPI: `/chat`, `/health`, `DELETE /sessions/{id}`; sessioni in memoria con LRU e lock; ID sessione nell'header `X-Session-Id` | Il corpo della risposta contiene esattamente i quattro campi della traccia |

## Strategia di memoria
Il trimming puro dimentica l'ordine citato al primo turno; la summarization pura costa una chiamata a ogni turno. Il summary buffer riassume solo quando la storia supera la soglia, in modo incrementale e con schema fisso. Gli order ID sono estratti con una regex, quindi non dipendono dal modello. Se il riassunto fallisce, la memoria degrada a trimming senza perdere gli ID.

## Valutazione
Dataset di 18 casi (conoscenza, ordini, escalation, memoria, multi-tool, fuori ambito). Controlli deterministici più RAGAS (Faithfulness, ContextPrecision) e DeepEval (GEval correctness, AnswerRelevancy). Giudice `openai/gpt-4.1-mini`, di una famiglia diversa dall'agente per evitare il bias di auto-valutazione.

| Metrica | v1 | v2 (prompt con regola 7 sull'escalation) |
|---|---|---|
| Faithfulness | 0.61 | 0.68 |
| Context precision | 0.89 | 0.89 |
| Correctness (GEval) | 0.74 | 0.81 |
| Answer relevancy | 0.84 | 0.85 |
| Correctness sui casi di escalation | 0.40 | 0.77 |
| Check deterministici superati | 18/18 | 17/18 e 17/18 (in due run, con un caso diverso che fallisce ogni volta) |

**Lettura dei risultati.** Differenze inferiori a circa 0.1 rientrano nel rumore tra esecuzioni, perché il modello non è deterministico anche a temperatura 0. Il miglioramento chiaro riguarda l'escalation (0.40 → 0.77), ed è coerente con la modifica fatta al prompt.

**Debolezze rimaste.**
- `kb_05`: faithfulness 0.0, perché il modello aggiunge dettagli non presenti nei documenti. La regola 2 non basta.
- `mem_02`: la domanda di follow-up vaga viene risposta senza richiamare il tool.
- `oos_02`: molte ricerche inutili e `confidence` instabile tra un'esecuzione e l'altra. Ho scelto di non ottimizzarla ulteriormente e di documentarla.
- `mt_02`: context precision 0.0, probabilmente un artefatto della metrica (non verificato).
- Answer relevancy penalizza i rifiuti corretti (`oos_01`), e faithfulness penalizza cortesie e numeri di ticket. Alcuni punteggi bassi sono quindi limiti delle metriche, non dell'agente.

## Limiti e lavoro futuro
- Sessioni in memoria: si perdono al riavvio. In produzione servirebbe Redis o un database.
- Nessuna autenticazione: chi conosce un ID ordine ne legge lo stato (nome e città non vengono esposti).
- Piano gratuito Groq: oltre i limiti di token l'API risponde 503.
- Il primo avvio scarica circa 2 GB di modello di embedding.
- Data di riferimento fissa (`REFERENCE_DATE`), per rendere riproducibili demo e test.
- Possibili miglioramenti: definire esplicitamente i livelli di `confidence` nel prompt di valutazione, verificare `mt_02` sui contesti recuperati, rendere strutturato il flag "pratica in attesa di operatore".